## Paper_Main — Unified NODE + RNN Pipeline (Base + Physics-Informed)

**Models:** NODE-ED | NODE-ED-Uncoupled | RNN  ×  {Base, PD, Collocation} = **9 total**

**Evaluation:**
1. Optuna (small-model, MSE objective)
2. 100 repeated 5-fold cross-validation — model selection + uncertainty
3. Single 5-fold: training loss curves + concentration profiles (standardised axis)
4. Computational cost profiling
5. Data efficiency ablation: 30→25→20→15

**Statistical significance:** Paired t-test + Cohen's d (100 repeated k-fold scores).

**Physics variants:** `_PD` = learnable kinetic params | `_CL` = collocation.
Kinetic parameter (kga/kla/Kga) mean ± std reported across repeated k-fold runs.

**Metrics:** MSE, RMSE, MAPE per species per timestep + aggregated. **No R².**

In [1]:
import sys, subprocess
for pkg in ["optuna", "torchdiffeq", "scikit-learn", "matplotlib"]:
    try:
        __import__(pkg.replace("-","_"))
    except ImportError:
        subprocess.check_call([sys.executable,"-m","pip","install",pkg,"--quiet"])
print("✅ Packages ready")


c:\Users\Guest2\Desktop\Felicia's\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Packages ready


In [2]:
import random, time, warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from scipy.stats import ttest_rel, shapiro
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from torchdiffeq import odeint

# ── Global config ─────────────────────────────────────────────────────────────
DEVICE      = "cpu"
GLOBAL_SEED = 42

DATASET_BASENAME   = "Final_Dataset_outputchange"
COLLOC_BASENAME    = "Collocation_Points"
RESULTS_DIR        = Path("Paper_Results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Model types ───────────────────────────────────────────────────────────────
BASE_MODELS   = ["NODE_ED", "NODE_ED_UNC", "RNN", "GRU"]

PHYSICS_MODELS = [
    "NODE_ED_PD",      "NODE_ED_UNC_PD",      "RNN_PD",  "GRU_PD",
    "NODE_ED_CL",      "NODE_ED_UNC_CL",      "RNN_CL",  "GRU_CL",
]

MODEL_TYPES = BASE_MODELS + PHYSICS_MODELS

PARAM_DISCOVERY_MODELS = {m for m in MODEL_TYPES if "_PD" in m}
COLLOCATION_MODELS     = {m for m in MODEL_TYPES if "_CL" in m}

# ── Data dimensions ───────────────────────────────────────────────────────────
N_INPUT           = 4
N_OUTPUT_FEATURES = 2
N_TIMESTEPS       = 4
TIME_GRID         = [4, 8, 12, 16]

# ── NODE integration ──────────────────────────────────────────────────────────
ODE_SOLVER = "rk4"
T_EVAL     = torch.tensor([0.25, 0.50, 0.75, 1.00], dtype=torch.float32)

# ── Optuna ────────────────────────────────────────────────────────────────────
N_TRIALS         = 50    # Stage-1 Optuna: architecture search
OPTUNA_FOLDS = 5

# ── Evaluation — 100 repeated k-fold ─────────────────────────────────────────
SINGLE_KFOLD_FOLDS     = 5
REPEATED_KFOLD_REPEATS = 100

# ── Plots — single k-fold with loss history + predictions ────────────────────
PLOT_KFOLD_SEED = GLOBAL_SEED   # seed for the single k-fold used for plots

# ── Data efficiency ablation ──────────────────────────────────────────────────
ABLATION_SIZES = [30, 25, 20, 15]

# ── Physics constants ─────────────────────────────────────────────────────────
Da           = 1.7998e-9
Db           = 2.13e-9
m_henry      = 1633.91
P_total      = 101325.0
C_total      = 5.55e4

KGA_INIT     = 3.2574e-5;  KGA_MIN    = KGA_INIT/10;      KGA_MAX    = KGA_INIT*10
KLA_INIT     = 0.03175;    KLA_MIN    = KLA_INIT/10;      KLA_MAX    = KLA_INIT*10
KGA_OVR_INIT = 8.0290e-6;  KGA_OVR_MIN= KGA_OVR_INIT/10; KGA_OVR_MAX= KGA_OVR_INIT*10

# ── Physics loss weights ─────────────────────────────────────────────────────
# Base defaults used only if a given model's Optuna config doesn't include them.
# (In the updated workflow, _PD/_CL variants are tuned by Optuna directly.)
PHYSICS_WEIGHT_DEFAULT = 0.1
# PHYSICS_WEIGHT_SWEEP removed — physics weight now optimised by Optuna per model
COLLOC_BATCH_SIZE      = 100


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(GLOBAL_SEED)
print(f"✅ Config loaded")
print(f"   Base models   : {BASE_MODELS}")
print(f"   Physics models: {PHYSICS_MODELS}")
print(f"   Evaluation    : {REPEATED_KFOLD_REPEATS} repeated × {SINGLE_KFOLD_FOLDS}-fold")


✅ Config loaded
   Base models   : ['NODE_ED', 'NODE_ED_UNC', 'RNN', 'GRU']
   Physics models: ['NODE_ED_PD', 'NODE_ED_UNC_PD', 'RNN_PD', 'GRU_PD', 'NODE_ED_CL', 'NODE_ED_UNC_CL', 'RNN_CL', 'GRU_CL']
   Evaluation    : 100 repeated × 5-fold


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# NODE-ED  (shared latent encoder-decoder)
# ─────────────────────────────────────────────────────────────────────────────
class ODEFuncLatent(nn.Module):
    def __init__(self, dim, hidden_dim=16, layers=2):
        super().__init__()
        net, in_d = [], dim + 1
        for _ in range(max(1, layers - 1)):
            net += [nn.Linear(in_d, hidden_dim), nn.Tanh()]
            in_d = hidden_dim
        net += [nn.Linear(in_d, dim)]
        self.net = nn.Sequential(*net)

    def forward(self, t, h):
        t_s = float(t.detach()) if torch.is_tensor(t) else float(t)
        return self.net(torch.cat([h, torch.full((h.shape[0],1), t_s, device=h.device)], -1))


class NODEEncoderDecoder(nn.Module):
    def __init__(self, n_input=4, n_output=2, latent_dim=16, ode_hidden=16, ode_layers=2,
                 solver=ODE_SOLVER):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(n_input, latent_dim), nn.Tanh(),
                                     nn.Linear(latent_dim, latent_dim))
        self.func    = ODEFuncLatent(latent_dim, ode_hidden, ode_layers)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, ode_hidden), nn.Tanh(),
                                     nn.Linear(ode_hidden, n_output))
        self.solver  = solver

    def forward(self, x, t_eval):
        h_traj = odeint(self.func, self.encoder(x), t_eval, method=self.solver).permute(1,0,2)
        return self.decoder(h_traj)


# ─────────────────────────────────────────────────────────────────────────────
# NODE-ED-Uncoupled  (independent latent ODE per species, shared encoder)
# ─────────────────────────────────────────────────────────────────────────────
class NODEEncoderDecoderUncoupled(nn.Module):
    """
    Hard-parameter sharing Multi-Task Learning NODE (Caruana, 1997).

    Architecture:
        Shared encoder  →  task-specific ODE (NaOH)  →  task-specific decoder (NaOH)
                        ↘  task-specific ODE (CO2)   →  task-specific decoder (CO2)

    Why this is correct MTL:
    - Shared encoder learns joint features useful for BOTH species from the same
      4 inputs — acts as regulariser at small N, captures chemical coupling
      (CO2 + 2NaOH → Na2CO3 + H2O: both species driven by same conditions)
    - Task-specific ODEs learn species-specific latent dynamics independently
    - Task-specific decoders produce species-specific concentration predictions

    Physics loss gradient path for PD/CL (CO2 only):
        phys_loss → decoder_co2 → func_co2 → h0 (shared encoder)
    NaOH ODE and decoder receive ZERO direct physics gradient.
    The shared encoder receives physics gradient but is regulated by l_naoh
    which pulls it toward NaOH-consistent features simultaneously — natural
    damping of the physics signal through multi-task competition.

    vs old "UNC" (two independent models):
        Old: encoder_naoh/co2 fully separate — no MTL, 2x params, no coupling
        New: shared encoder — true MTL, ~1.5x NODE_ED params, captures coupling
    """
    def __init__(self, n_input=4, latent_dim=16, ode_hidden=16, ode_layers=2,
                 solver=ODE_SOLVER):
        super().__init__()
        # Shared encoder — joint feature extraction for both species
        self.encoder      = nn.Sequential(nn.Linear(n_input, latent_dim), nn.Tanh(),
                                          nn.Linear(latent_dim, latent_dim))
        # Task-specific ODE per species
        self.func_naoh    = ODEFuncLatent(latent_dim, ode_hidden, ode_layers)
        self.func_co2     = ODEFuncLatent(latent_dim, ode_hidden, ode_layers)
        # Task-specific decoder per species
        self.decoder_naoh = nn.Sequential(nn.Linear(latent_dim, ode_hidden), nn.Tanh(),
                                          nn.Linear(ode_hidden, 1))
        self.decoder_co2  = nn.Sequential(nn.Linear(latent_dim, ode_hidden), nn.Tanh(),
                                          nn.Linear(ode_hidden, 1))
        self.solver = solver

    def forward(self, x, t_eval):
        h0     = self.encoder(x)                                    # shared representation
        h_naoh = odeint(self.func_naoh, h0,
                        t_eval, method=self.solver).permute(1,0,2)  # NaOH latent trajectory
        h_co2  = odeint(self.func_co2,  h0,
                        t_eval, method=self.solver).permute(1,0,2)  # CO2 latent trajectory
        return torch.cat([self.decoder_naoh(h_naoh),
                          self.decoder_co2(h_co2)], dim=-1)         # (batch, T, 2)


# ─────────────────────────────────────────────────────────────────────────────
# RNN
# ─────────────────────────────────────────────────────────────────────────────
class _SeqRecurrentBase(nn.Module):
    """
    Shared base for RNN and GRU with:
    - Time encoding: normalised linspace(0,1,T) appended at each step
    - LayerNorm on ALL hidden states for stability
    - Per-timestep output heads: each timestep's hidden state maps to its own prediction
    - Separate species heads (head_naoh, head_co2) — each Linear(n_hidden, 1)

    Key design: uses ALL hidden states (out[:, :, :]) not just the last one.
    This means:
    - t=4 prediction comes from h(t=4), t=16 from h(t=16)
    - Physics loss at t=16 has direct gradient only to h(t=16)
    - Gradient to earlier steps flows through RNN sequential processing (attenuated)
    - Reduces physics contamination vs old design where one hidden state → all predictions

    Literature: Cho et al. (2014) — GRU; Elman (1990) — RNN.
    Per-timestep decoding is standard in seq2seq (Sutskever et al. 2014).
    """
    def __init__(self, n_input, n_output_features, n_timesteps,
                 n_hidden=16, n_layers=1, dropout=0.0):
        super().__init__()
        self.n_timesteps       = n_timesteps
        self.n_output_features = n_output_features
        self._build_rnn(n_input + 1, n_hidden, n_layers, dropout)
        self.norm      = nn.LayerNorm(n_hidden)
        # Per-timestep heads: Linear(n_hidden → 1) applied at each step
        self.head_naoh = nn.Linear(n_hidden, 1)
        self.head_co2  = nn.Linear(n_hidden, 1)

    def _build_rnn(self, input_size, n_hidden, n_layers, dropout):
        raise NotImplementedError

    def forward(self, x):
        batch = x.size(0)
        t = torch.linspace(0, 1, self.n_timesteps, device=x.device)
        t = t.view(1, self.n_timesteps, 1).expand(batch, -1, -1)
        x_seq = x.unsqueeze(1).expand(-1, self.n_timesteps, -1)
        x_seq = torch.cat([x_seq, t], dim=-1)          # (batch, T, n_input+1)
        out, _ = self.rnn(x_seq)                        # (batch, T, n_hidden)
        out    = self.norm(out)                         # LayerNorm on ALL steps
        naoh   = self.head_naoh(out)                    # (batch, T, 1)
        co2    = self.head_co2(out)                     # (batch, T, 1)
        return torch.cat([naoh, co2], dim=-1)           # (batch, T, 2)


class SeqRNN(_SeqRecurrentBase):
    """Simple RNN (tanh) — subclass of shared base."""
    def _build_rnn(self, input_size, n_hidden, n_layers, dropout):
        self.rnn = nn.RNN(input_size, n_hidden, n_layers, batch_first=True,
                          nonlinearity="tanh",
                          dropout=dropout if n_layers > 1 else 0.0)


class SeqGRU(_SeqRecurrentBase):
    """
    GRU — same architecture as SeqRNN but uses nn.GRU.
    Shares: time encoding, LayerNorm, separate species heads.
    GRU gating improves gradient flow vs vanilla RNN.
    Note: same NaOH contamination issue as RNN (shared hidden state).
    """
    def _build_rnn(self, input_size, n_hidden, n_layers, dropout):
        self.rnn = nn.GRU(input_size, n_hidden, n_layers, batch_first=True,
                          dropout=dropout if n_layers > 1 else 0.0)


def build_model(model_type, params):
    arch = _base_arch(model_type)
    if arch == "NODE_ED":
        return NODEEncoderDecoder(
            latent_dim=int(params.get("latent_dim",16)),
            ode_hidden=int(params.get("ode_hidden",16)),
            ode_layers=int(params.get("ode_layers",2)))
    elif arch == "NODE_ED_UNC":
        return NODEEncoderDecoderUncoupled(
            latent_dim=int(params.get("latent_dim",16)),
            ode_hidden=int(params.get("ode_hidden",16)),
            ode_layers=int(params.get("ode_layers",2)))
    elif arch == "RNN":
        return SeqRNN(N_INPUT, N_OUTPUT_FEATURES, N_TIMESTEPS,
                      n_hidden=int(params.get("n_hidden",16)),
                      n_layers=int(params.get("n_layers",1)),
                      dropout=float(params.get("dropout",0.0)))
    elif arch == "GRU":
        return SeqGRU(N_INPUT, N_OUTPUT_FEATURES, N_TIMESTEPS,
                      n_hidden=int(params.get("n_hidden",16)),
                      n_layers=int(params.get("n_layers",1)),
                      dropout=float(params.get("dropout",0.0)))
    raise ValueError(f"Unknown model_type/arch: {model_type}")

print("✅ Model definitions loaded (NODE_ED, NODE_ED_UNC, RNN, GRU + physics variants)")


✅ Model definitions loaded (NODE_ED, NODE_ED_UNC, RNN, GRU + physics variants)


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Data loading
# ─────────────────────────────────────────────────────────────────────────────
def _norm_key(n): return "".join(ch.lower() for ch in str(n) if ch.isalnum())
def _pick_col(df, aliases):
    m = {_norm_key(c):c for c in df.columns}
    for a in aliases:
        if _norm_key(a) in m: return m[_norm_key(a)]
    return None

def find_dataset_path():
    for c in [f"{DATASET_BASENAME}.csv", f"{DATASET_BASENAME}_t16.csv",
              "Final_Dataset_outputchange_t16.csv", f"{DATASET_BASENAME}.xlsx"]:
        if Path(c).exists(): return Path(c)
    raise FileNotFoundError(f"Dataset not found. Tried variations of {DATASET_BASENAME}")

def load_outputchange_data(verbose=True, path=None):
    p  = Path(path) if path else find_dataset_path()
    df = pd.read_csv(p) if p.suffix.lower()!=".xlsx" else pd.read_excel(p)
    df = df.dropna(how="all",axis=0).dropna(how="all",axis=1)

    # Input columns: NaOH outlet t=0, CO2 outlet t=0, liquid inlet, gas inlet
    # Supports both Final_Dataset_outputchange and dynamic_state column naming
    c0 = _pick_col(df,["Liquid Outlet_0","NaOH Outlet_0","XB4"])
    c1 = _pick_col(df,["Gas Outlet_0","CO2 Outlet_0","YA1"])
    c2 = _pick_col(df,["Final_Liquid Inlet","New_Liquid Inlet","LiquidInlet","Qw"])
    c3 = _pick_col(df,["Final_Gas Inlet","New_Gas Inlet","GasInlet","Qg"])
    if any(c is None for c in [c0,c1,c2,c3]):
        raise ValueError(f"Cannot map input columns. Available: {list(df.columns)}")

    # Output columns: NaOH and CO2 outlet at t=4,8,12,16 min
    out_cols=[]
    for t in [4,8,12,16]:
        a=_pick_col(df,[f"Liquid Outlet_{t}",f"NaOH Outlet_{t}",f"{t} XB4",f"{t}XB4"])
        b=_pick_col(df,[f"Gas Outlet_{t}",f"CO2 Outlet_{t}",f"{t} YA4",f"{t}YA4"])
        if a is None or b is None: raise ValueError(f"Missing output cols t={t}. Available: {list(df.columns)}")
        out_cols+=[a,b]

    X     = df[[c0,c1,c2,c3]].to_numpy(np.float32)
    y_raw = df[out_cols].to_numpy(np.float32)
    y     = y_raw.reshape(-1,N_TIMESTEPS,N_OUTPUT_FEATURES)
    if verbose:
        print("="*60)
        print(f"Dataset : {p}"); print(f"Samples : {X.shape[0]}")
        print(f"Inputs  : {[c0,c1,c2,c3]}")
        print(f"Outputs : {out_cols}")
        print(f"X shape : {X.shape}  y shape: {y.shape}"); print("="*60)
    return X,y,p,[c0,c1,c2,c3],out_cols

def create_fold_indices(X,n_folds=5,random_state=42):
    kf=KFold(n_splits=n_folds,shuffle=True,random_state=random_state)
    return [(tr,te) for tr,te in kf.split(X)]

def normalize_data(X_tr,y_tr,X_te,y_te):
    xm=X_tr.mean(0,keepdims=True); xs=X_tr.std(0,keepdims=True)+1e-8
    ym=y_tr.reshape(-1,2).mean(0,keepdims=True)
    ys=y_tr.reshape(-1,2).std(0,keepdims=True)+1e-8
    Xtn=(X_tr-xm)/xs; Xten=(X_te-xm)/xs
    ytn=y_tr.copy(); yten=y_te.copy()
    for j in range(2):
        ytn[:,:,j]=(y_tr[:,:,j]-ym[0,j])/ys[0,j]
        yten[:,:,j]=(y_te[:,:,j]-ym[0,j])/ys[0,j]
    return Xtn,ytn,Xten,yten,xm,xs,ym,ys

def to_tensors(X_tr,y_tr,X_te,y_te,device="cpu"):
    Xtn,ytn,Xten,yten,xm,xs,ym,ys=normalize_data(X_tr,y_tr,X_te,y_te)
    dev=torch.device(device)
    return (torch.tensor(Xtn,dtype=torch.float32,device=dev),
            torch.tensor(ytn,dtype=torch.float32,device=dev),
            torch.tensor(Xten,dtype=torch.float32,device=dev),
            torch.tensor(yten,dtype=torch.float32,device=dev),
            {"x_mean":xm,"x_std":xs,"y_mean":ym,"y_std":ys})

# ─────────────────────────────────────────────────────────────────────────────
# Metrics: MSE, RMSE, MAPE per species per timestep + aggregated  (no R²)
# ─────────────────────────────────────────────────────────────────────────────
SPECIES = ["naoh","co2"]
EPS = 1e-8

def compute_metrics(y_true, y_pred):
    """
    y_true, y_pred: (N, T, 2) physical units
    Returns flat dict of per-species/per-timestep + aggregated MSE/RMSE/MAPE.
    """
    out={}
    for si,sp in enumerate(SPECIES):
        yt=y_true[:,:,si]; yp=y_pred[:,:,si]
        for ti,t in enumerate(TIME_GRID):
            e=yt[:,ti]-yp[:,ti]
            out[f"{sp}_t{t}_mse"]  = float(np.mean(e**2))
            out[f"{sp}_t{t}_rmse"] = float(np.sqrt(out[f"{sp}_t{t}_mse"]))
            out[f"{sp}_t{t}_mape"] = float(np.mean(np.abs(e)/(np.abs(yt[:,ti])+EPS))*100)
        e_all=( yt-yp).ravel(); yt_all=yt.ravel()
        out[f"{sp}_mse"]  = float(np.mean(e_all**2))
        out[f"{sp}_rmse"] = float(np.sqrt(out[f"{sp}_mse"]))
        out[f"{sp}_mape"] = float(np.mean(np.abs(e_all)/(np.abs(yt_all)+EPS))*100)
    eg=( y_true-y_pred).ravel(); ytg=y_true.ravel()
    out["agg_mse"]  = float(np.mean(eg**2))
    out["agg_rmse"] = float(np.sqrt(out["agg_mse"]))
    out["agg_mape"] = float(np.mean(np.abs(eg)/(np.abs(ytg)+EPS))*100)
    return out

def denorm(pred_n, true_n, ym, ys):
    p=pred_n.copy(); tr=true_n.copy()
    for j in range(2):
        p[:,:,j]=pred_n[:,:,j]*ys[0,j]+ym[0,j]
        tr[:,:,j]=true_n[:,:,j]*ys[0,j]+ym[0,j]
    return p,tr

# ─────────────────────────────────────────────────────────────────────────────
# train_one_split
# ─────────────────────────────────────────────────────────────────────────────
print("✅ Data helpers + metrics loaded (train_one_split is in Cell 6)")


✅ Data helpers + metrics loaded (train_one_split is in Cell 6)


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Physics constants, governing equations, KineticParams, collocation loader
# ─────────────────────────────────────────────────────────────────────────────
from scipy.optimize import brentq

class KineticParams(nn.Module):
    """
    Learnable mass-transfer kinetic parameters kga, kla, Kga stored in log-space.
    Hard-clamped to ±1 order of magnitude from initial values.
    """
    def __init__(self):
        super().__init__()
        self.log_kga = nn.Parameter(torch.tensor(float(np.log(KGA_INIT)),     dtype=torch.float32))
        self.log_kla = nn.Parameter(torch.tensor(float(np.log(KLA_INIT)),     dtype=torch.float32))
        self.log_Kga = nn.Parameter(torch.tensor(float(np.log(KGA_OVR_INIT)), dtype=torch.float32))

    def clamp_(self):
        with torch.no_grad():
            self.log_kga.clamp_(float(np.log(KGA_MIN)),     float(np.log(KGA_MAX)))
            self.log_kla.clamp_(float(np.log(KLA_MIN)),     float(np.log(KLA_MAX)))
            self.log_Kga.clamp_(float(np.log(KGA_OVR_MIN)), float(np.log(KGA_OVR_MAX)))

    def values(self):
        return (float(torch.exp(self.log_kga).detach()),
                float(torch.exp(self.log_kla).detach()),
                float(torch.exp(self.log_Kga).detach()))


def load_colloc_data(path=None, verbose=True):
    """
    Load collocation point grid CSV.
    Returns X_colloc (M,4) and ya4_colloc (M,) for physics constraint training.
    """
    if path is None:
        for c in [f"{COLLOC_BASENAME}.csv", COLLOC_BASENAME,
                  "Collocation_Points.csv"]:
            if Path(c).exists():
                path = Path(c); break
        if path is None:
            raise FileNotFoundError(f"Collocation grid not found. Tried {COLLOC_BASENAME}.csv")
    path = Path(path)
    df   = pd.read_csv(path) if path.suffix.lower()!=".xlsx" else pd.read_excel(path)
    df   = df.dropna(how="all",axis=0).dropna(how="all",axis=1)

    col_liq = _pick_col(df,["Final_Liquid Inlet-SIM","Final_Liquid_Inlet_SIM"])
    col_gas = _pick_col(df,["Final_Gas Inlet-SIM",  "Final_Gas_Inlet_SIM"])
    col_ya4 = _pick_col(df,["Gas Outlet_16-SIM",    "Gas_Outlet_16_SIM","YA4"])
    if any(c is None for c in [col_liq,col_gas,col_ya4]):
        raise ValueError(f"Cannot map colloc columns. Available: {list(df.columns)}")

    M           = len(df)
    X_colloc    = np.zeros((M,4), dtype=np.float32)
    X_colloc[:,2] = df[col_liq].to_numpy(np.float32)
    X_colloc[:,3] = df[col_gas].to_numpy(np.float32)
    ya4_colloc    = df[col_ya4].to_numpy(np.float32)

    if verbose:
        print(f"  Collocation grid: {M} rows  ya4∈[{ya4_colloc.min():.4f},{ya4_colloc.max():.4f}]")
    return X_colloc, ya4_colloc

print("✅ Physics constants + governing equations + KineticParams + colloc loader loaded")


def _gov_height_torch(ya4, ya1, xb4, G, L, kga, kla, Kga):
    """
    Differentiable PyTorch version of _gov_height.
    All tensors — gradient flows to kga, kla, Kga and pred_ya4.
    Clamps prevent log(<=0) numerical instability.
    """
    eps = 1e-12
    P = 101325.0; Da = 1.7998e-9; Db = 2.13e-9; m = 1633.91; C = 5.55e4

    beta_one = (L * P * Da * kga) / (G * C * Db * kla + eps)
    beta_two = (P * Da * kga * 2.0) / (C * Db * kla + eps)
    XBC      = (beta_one * xb4 + beta_two * ya4) / (1.0 + beta_one + eps)
    YA3      = (ya4 + (L / (2.0 * G + eps)) * (xb4 - XBC)).clamp(min=eps)
    YA2      = (ya4 + (L / (2.0 * G + eps)) * xb4).clamp(min=eps)

    h3 = (G / (kga * P + eps)) * torch.log((YA3 / ya4.clamp(min=eps)).clamp(min=eps))

    alpha  = 1.0 - (Db * m * G) / (Da * L + eps)
    beta_v = (Db * m * XBC) / (Da * 2.0) + (Db * m * G * YA3) / (Da * L + eps)
    num    = (alpha * YA2 + beta_v).clamp(min=eps)
    den    = (alpha * YA3 + beta_v).clamp(min=eps)
    denom_h2 = (Kga * P * (alpha.abs() + eps)).abs() + eps
    h2     = (G / denom_h2) * torch.log((num / den).clamp(min=eps))

    ratio   = 1.0 - m * G / (L + eps)
    log_arg = (ratio * (ya1 / YA2.clamp(min=eps)) + m * G / (L + eps)).clamp(min=eps)
    denom_h1 = (Kga * P * (ratio.abs() + eps)).abs() + eps
    h1      = (G / denom_h1) * torch.log(log_arg)

    return h1 + h2 + h3


✅ Physics constants + governing equations + KineticParams + colloc loader loaded


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Unified train_one_split — handles base + param discovery + colloc + combined
#
# model_type convention:
#   NODE_ED, NODE_ED_UNC, RNN         → base (no physics)
#   NODE_ED_PD, NODE_ED_UNC_PD, RNN_PD → + governing equation param discovery
#   NODE_ED_CL, NODE_ED_UNC_CL, RNN_CL → + collocation constraint
#
# The base architecture is derived from the model_type by stripping the suffix.
# Physics losses are activated based on whether _PD or _CL is in the suffix.
# ─────────────────────────────────────────────────────────────────────────────
def _base_arch(model_type):
    """Strip physics suffix to get base architecture name."""
    for suffix in ("_PD","_CL"):
        if model_type.endswith(suffix):
            return model_type[:-len(suffix)]
    return model_type


def train_one_split(X_train, y_train, X_test, y_test,
                    X_train_raw=None, X_test_raw=None,
                    Qg_train_raw=None, Qw_train_raw=None,
                    Qg_test_raw=None,  Qw_test_raw=None,
                    X_colloc=None, ya4_colloc=None,
                    model_type="NODE_ED", params=None, seed=42, device="cpu",
                    return_loss_history=False, return_predictions=False):
    """
    Unified training function for all model variants.

    Physics losses activated automatically based on model_type suffix:
      _PD : governing-equation loss with learnable kga, kla, Kga
      _CL : collocation constraint loss (100 random rows per epoch)

    Extra arguments required for physics variants:
      X_train_raw, X_test_raw : un-normalised inputs (cols 2=XB4, 3=YA1)
      Qg_train_raw, Qw_train_raw, Qg_test_raw, Qw_test_raw : flow rates
      X_colloc, ya4_colloc : collocation grid (for _CL variants)
    """
    if params is None: params = {}
    set_seed(seed)
    t0      = time.perf_counter()
    max_ep  = int(params.get("max_epochs", 300))
    use_pd  = model_type in PARAM_DISCOVERY_MODELS
    use_cl  = model_type in COLLOCATION_MODELS
    arch    = _base_arch(model_type)
    is_node = arch in ("NODE_ED","NODE_ED_UNC")

    Xtr,ytr,Xte,yte,norm = to_tensors(X_train,y_train,X_test,y_test,device=device)
    ym,ys  = norm["y_mean"],norm["y_std"]
    co2_std  = float(ys[0,1]); co2_mean = float(ym[0,1])
    model  = build_model(arch, params).to(device)

    # ── Parameter groups ──────────────────────────────────────────────────────
    param_groups = [{"params": model.parameters(), "weight_decay": float(params.get("weight_decay",1e-4))}]
    kinetic_module = None
    if use_pd:
        kinetic_module = KineticParams().to(device)
        param_groups.append({"params": kinetic_module.parameters(),
                              "weight_decay": float(params.get("kinetic_weight_decay",1.0))})

    opt    = torch.optim.AdamW(param_groups, lr=float(params.get("learning_rate",1e-3)))
    mse_fn = nn.MSELoss()
    t_ev   = T_EVAL.to(Xtr.device)

    # ── Raw inputs for governing equation ─────────────────────────────────────
    # Disable PD loss silently if any required raw input is missing
    _pd_inputs_ok = (use_pd and X_train_raw is not None
                     and Qg_train_raw is not None and Qw_train_raw is not None)
    if use_pd and not _pd_inputs_ok:
        warnings.warn(f"{model_type}: Qg/Qw raw inputs missing — governing-equation loss disabled")
    if _pd_inputs_ok:
        # X columns: [0]=NaOH outlet t=0 (XB4), [1]=CO2 outlet t=0 (YA1),
        #            [2]=Liquid Inlet (Qw already in Qg_train_raw/Qw_train_raw)
        # Governing equation needs:
        #   YA1 = CO2 inlet mole ratio  → X column index 1 (Gas Outlet_0 / YA1)
        #   XB4 = NaOH inlet mole ratio → X column index 0 (Liquid Outlet_0 / XB4)
        #   Qg, Qw are passed separately via Qg_train_raw / Qw_train_raw
        # X column order: [0]=NaOH_out_t0, [1]=CO2_out_t0,
#                  [2]=NaOH_inlet (XB4), [3]=CO2_inlet (YA1)
        # Governing equation needs INLET mole ratios (cols 2,3)
        # NOT outlet values (cols 0,1) — this was the original bug
        XB4_tr = X_train_raw[:,2].astype(np.float32)   # NaOH inlet = col 2
        YA1_tr = X_train_raw[:,3].astype(np.float32)   # CO2  inlet = col 3
        Qg_tr  = Qg_train_raw.astype(np.float32)
        Qw_tr  = Qw_train_raw.astype(np.float32)
        XB4_te = X_test_raw[:,2].astype(np.float32)
        YA1_te = X_test_raw[:,3].astype(np.float32)
        Qg_te  = Qg_test_raw.astype(np.float32)
        Qw_te  = Qw_test_raw.astype(np.float32)

    # ── Collocation tensors ───────────────────────────────────────────────────
    X_colloc_t = ya4_t = None
    if use_cl and X_colloc is not None:
        xm = norm["x_mean"]; xs_n = norm["x_std"]
        # X_colloc cols 0,1 are correctly zero — the simulation assumes the
        # column starts empty (zero outlet concentrations at t=0).
        # Cols 2,3 hold the liquid/gas inlet flowrates from the colloc grid.
        X_colloc_n = (X_colloc - xm) / xs_n
        dev = torch.device(device)
        X_colloc_t = torch.tensor(X_colloc_n, dtype=torch.float32, device=dev)
        # Normalise ya4_colloc into the same z-score space as the model output.
        # ya4_colloc is in physical units (mol/L). Without normalisation the
        # collocation loss is ~1e-4 vs trajectory loss ~0.1 — a 1000× scale
        # mismatch that makes physics_weight effectively zero regardless of value.
        co2_mean_colloc = float(norm["y_mean"][0, 1])
        co2_std_colloc  = float(norm["y_std"][0, 1])
        ya4_colloc_norm = (ya4_colloc - co2_mean_colloc) / (co2_std_colloc + 1e-8)
        ya4_t      = torch.tensor(ya4_colloc_norm, dtype=torch.float32, device=dev)

    ph_w  = float(params.get("physics_weight",  PHYSICS_WEIGHT_DEFAULT))
    cl_w  = float(params.get("colloc_weight",   PHYSICS_WEIGHT_DEFAULT))
    nh_hist=[]; nc_hist=[]

    _t_train0     = time.perf_counter()

    for ep in range(max_ep):
        model.train(); opt.zero_grad()
        if use_pd and kinetic_module: kinetic_module.train()

        pred     = model(Xtr,t_ev) if is_node else model(Xtr)
        l_naoh   = mse_fn(pred[:,:,0], ytr[:,:,0])
        l_co2    = mse_fn(pred[:,:,1], ytr[:,:,1])
        total    = l_naoh + l_co2

        # ── Param discovery loss ──────────────────────────────────────────────
        # The governing equation is evaluated with current kinetic parameter
        # values (detached from the graph — numpy computation).
        # Gradients flow into kinetic parameters via TWO mechanisms:
        #   1. kga_v, kla_v, Kga_v are used to compute ya4_gov (the target).
        #      Since ya4_gov is numpy, the gradient w.r.t. kinetic params
        #      from this path is zero — the target is a "soft label".
        #   2. A direct regularisation loss on the log-parameters pulls them
        #      away from their initial values and allows AdamW to update them.
        #
        # The primary driver of kinetic parameter change is therefore:
        #   - The governing equation residual penalises the MODEL prediction
        #     so the model learns to match the physics-derived targets.
        #   - Kinetic params are updated by minimising the mismatch between
        #     the model's CO2 prediction and the governing equation target,
        #     mediated through the gradient of pred_co2_phys w.r.t. kga/kla/Kga
        #     via the chain rule through the physics loss.
        #
        # To make this work correctly, ya4_gov must be recomputed each epoch
        # with the CURRENT kinetic values so the target shifts as params update.
        if _pd_inputs_ok and kinetic_module is not None:
            # ── Option 1: Fully differentiable PD loss ────────────────────────
            # Plug model CO2 prediction into governing equation and penalise
            # height residual from TARGET_HEIGHT (0.35 m).
            # Gradient flows through _gov_height_torch to BOTH model weights
            # AND kinetic parameters. No .detach() — real gradients everywhere.
            # No kinetic_reg — phys_loss is the sole kinetic gradient source.
            kga = torch.exp(kinetic_module.log_kga)
            kla = torch.exp(kinetic_module.log_kla)
            Kga = torch.exp(kinetic_module.log_Kga)

            Qg_t  = torch.tensor(Qg_tr,  dtype=torch.float32, device=Xtr.device)
            Qw_t  = torch.tensor(Qw_tr,  dtype=torch.float32, device=Xtr.device)
            YA1_t = torch.tensor(YA1_tr, dtype=torch.float32, device=Xtr.device)
            XB4_t = torch.tensor(XB4_tr, dtype=torch.float32, device=Xtr.device)

            G_t = ((Qg_t / 60) * 1000 * 0.0011839) / (28.96 * 0.002027)
            L_t = ((Qw_t / 60) * 1000) / (18 * 0.002027)

            # Model CO2 at t=16 denormalised — clamped to physical positive range
            pred_ya4 = (pred[:, -1, 1] * co2_std + co2_mean).clamp(min=1e-10)

            h_pred = _gov_height_torch(
                pred_ya4, YA1_t, XB4_t, G_t, L_t, kga, kla, Kga)

            valid_mask = torch.isfinite(h_pred)
            if valid_mask.sum() > 0:
                TARGET_HEIGHT = 0.35
                # Normalise: loss = MSE((h_pred / 0.35), 1) so scale is O(1)
                phys_loss = mse_fn(
                    h_pred[valid_mask] / TARGET_HEIGHT,
                    torch.ones(valid_mask.sum(), device=Xtr.device))
                total = total + ph_w * phys_loss
            kinetic_module.clamp_()

        # ── Collocation loss ──────────────────────────────────────────────────
        if use_cl and X_colloc_t is not None:
            perm        = torch.randperm(X_colloc_t.size(0), device=Xtr.device)[:COLLOC_BATCH_SIZE]
            pred_col    = model(X_colloc_t[perm],t_ev) if is_node else model(X_colloc_t[perm])
            # pred_col[:,-1,1] is already in normalised space.
            # ya4_t is also normalised (see setup above).
            # Both on same scale → colloc_weight is a meaningful fraction.
            pred_co2_cl = pred_col[:,-1,1]
            total       = total + cl_w * mse_fn(pred_co2_cl, ya4_t[perm])

        total.backward(); opt.step()
        if use_pd and kinetic_module: kinetic_module.clamp_()
        if return_loss_history:
            nh_hist.append(float(l_naoh.detach()))
            nc_hist.append(float(l_co2.detach()))

    # ── Eval ──────────────────────────────────────────────────────────────────
    model.eval()
    if use_pd and kinetic_module: kinetic_module.eval()
    ev0=time.perf_counter()
    with torch.no_grad():
        trpn=(model(Xtr,t_ev) if is_node else model(Xtr)).cpu().numpy()
        tepn=(model(Xte,t_ev) if is_node else model(Xte)).cpu().numpy()
    ev_t=float(time.perf_counter()-ev0)

    # Metrics computed in NORMALISED space — consistent with training objective
    # and ensures NaOH/CO2 contribute equally regardless of concentration scale
    trpn_np = trpn;              tep_np = tepn
    trtn_np = ytr.cpu().numpy(); ten_np = yte.cpu().numpy()
    train_m = compute_metrics(trtn_np, trpn_np)
    test_m  = compute_metrics(ten_np,  tep_np)

    # Denormalised predictions kept for concentration profile plots only
    tp,tt=denorm(tepn,yte.cpu().numpy(),ym,ys)
    trp,trt=denorm(trpn,ytr.cpu().numpy(),ym,ys)
    m={}
    for k,v in train_m.items(): m[f"train_{k}"]=v
    for k,v in test_m.items():  m[f"test_{k}" ]=v

    # ── Learnt kinetic parameters ─────────────────────────────────────────────
    if use_pd and kinetic_module is not None:
        kga_f,kla_f,Kga_f = kinetic_module.values()
        m["final_kga"] = kga_f
        m["final_kla"] = kla_f
        m["final_Kga"] = Kga_f

    _t_train_total          = float(time.perf_counter() - _t_train0)
    m["train_time_sec"]     = _t_train_total
    m["avg_epoch_time_sec"] = _t_train_total / max(max_ep, 1)
    m["std_epoch_time_sec"] = 0.0
    m["eval_time_sec"]=ev_t
    m["split_total_time_sec"]=float(time.perf_counter()-t0)
    m["epochs"]=int(max_ep)

    out={"metrics":m}
    if return_loss_history: out["loss_naoh"]=nh_hist; out["loss_co2"]=nc_hist
    if return_predictions:  out["te_pred"]=tp; out["te_true"]=tt
    return out

print("✅ Unified train_one_split loaded (base + PD + CL + CB variants)")


✅ Unified train_one_split loaded (base + PD + CL + CB variants)


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Data loading helper
# ─────────────────────────────────────────────────────────────────────────────
def load_physics_inputs(X_data, y_data, dataset_path):
    df = pd.read_csv(dataset_path) if str(dataset_path).endswith('.csv') else pd.read_excel(dataset_path)
    df = df.dropna(how="all",axis=0).dropna(how="all",axis=1)
    col_qg = _pick_col(df,["Qg","qg","GasFlow","Q_gas","GasFlowrate","Final_Gas Inlet",
                            "New_Gas Inlet","GasInlet"])
    col_qw = _pick_col(df,["Qw","qw","LiquidFlow","Q_liquid","LiquidFlowrate",
                            "Final_Liquid Inlet","New_Liquid Inlet","LiquidInlet"])
    Qg_data = df[col_qg].to_numpy(np.float32) if col_qg else None
    Qw_data = df[col_qw].to_numpy(np.float32) if col_qw else None
    if Qg_data is None:
        print("  ⚠ Qg column not found — governing-equation loss disabled for _PD variants")
    if Qw_data is None:
        print("  ⚠ Qw column not found — governing-equation loss disabled for _PD variants")
    return X_data.copy(), Qg_data, Qw_data


# ─────────────────────────────────────────────────────────────────────────────
# Optuna hyperparameter search
# ─────────────────────────────────────────────────────────────────────────────
def optuna_search_model(X_data, y_data, model_type="NODE_ED",
                        X_data_raw=None, Qg_data=None, Qw_data=None,
                        X_colloc=None, ya4_colloc=None,
                        n_trials=N_TRIALS, n_folds=OPTUNA_FOLDS,
                        seed=GLOBAL_SEED, device=DEVICE,
                        max_epochs_choices=None):
    folds    = create_fold_indices(X_data, n_folds=n_folds, random_state=seed)
    use_pd   = model_type in PARAM_DISCOVERY_MODELS
    use_cl   = model_type in COLLOCATION_MODELS
    arch     = _base_arch(model_type)

    max_epoch_space = [200, 300, 500] if max_epochs_choices is None else list(max_epochs_choices)

    def objective(trial):
        t0=time.perf_counter()
        if arch in ("NODE_ED","NODE_ED_UNC"):
            p={"ode_hidden": trial.suggest_int("ode_hidden",8,128),
               "ode_layers": trial.suggest_int("ode_layers",1,3),
               "latent_dim": trial.suggest_int("latent_dim",8,128)}
        elif arch == "RNN":
            p={"n_hidden": trial.suggest_int("n_hidden",8,128),
               "n_layers": trial.suggest_int("n_layers",1,3),
               "dropout":  trial.suggest_float("dropout",0.0,0.3)}
        else:  # GRU — same search space as RNN
            p={"n_hidden": trial.suggest_int("n_hidden",8,128),
               "n_layers": trial.suggest_int("n_layers",1,3),
               "dropout":  trial.suggest_float("dropout",0.0,0.3)}
        p["learning_rate"]=trial.suggest_float("learning_rate",1e-4,5e-3,log=True)
        p["weight_decay"] =trial.suggest_float("weight_decay", 1e-6,1e-2,log=True)
        p["max_epochs"]   =trial.suggest_categorical("max_epochs", max_epoch_space)
        if use_pd:
            p["physics_weight"]      =trial.suggest_float("physics_weight",1e-3,1.0,log=True)
            p["kinetic_weight_decay"]=trial.suggest_float("kinetic_weight_decay",1e-2,100.0,log=True)
        if use_cl:
            p["colloc_weight"]=trial.suggest_float("colloc_weight",1e-3,1.0,log=True)

        scores=[]
        for fid,(tr,te) in enumerate(folds,1):
            kw={}
            if use_pd or use_cl:
                kw.update({"X_train_raw": X_data_raw[tr] if X_data_raw is not None else None,
                           "X_test_raw":  X_data_raw[te] if X_data_raw is not None else None,
                           "Qg_train_raw":Qg_data[tr]    if Qg_data    is not None else None,
                           "Qw_train_raw":Qw_data[tr]    if Qw_data    is not None else None,
                           "Qg_test_raw": Qg_data[te]    if Qg_data    is not None else None,
                           "Qw_test_raw": Qw_data[te]    if Qw_data    is not None else None})
            if use_cl:
                kw.update({"X_colloc":X_colloc,"ya4_colloc":ya4_colloc})
            res=train_one_split(X_data[tr],y_data[tr],X_data[te],y_data[te],
                                model_type=model_type,params=p,
                                seed=seed+trial.number*100+fid,device=device,**kw)
            scores.append(-res["metrics"]["test_agg_mse"])
            trial.report(float(np.mean(scores)),step=fid)
            if trial.should_prune(): raise optuna.exceptions.TrialPruned()
        trial.set_user_attr("trial_duration_sec",float(time.perf_counter()-t0))
        return float(np.mean(scores))

    study=optuna.create_study(direction="maximize",
                              sampler=optuna.samplers.TPESampler(seed=seed),
                              pruner=optuna.pruners.MedianPruner(n_startup_trials=10))
    study.optimize(objective,n_trials=n_trials,show_progress_bar=False)
    rows=[{"number":t.number,"state":str(t.state).split(".")[-1],
           "neg_mse_cv":t.value,
           "trial_duration_sec":t.user_attrs.get("trial_duration_sec",np.nan),
           **t.params} for t in study.trials]
    return study.best_params, study.best_value, pd.DataFrame(rows)


# ─────────────────────────────────────────────────────────────────────────────
# Stage-2 Optuna: optimise physics/colloc weight only, architecture fixed
# ─────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────
# Single k-fold (used for plots — loss curves + concentration profiles)
# ─────────────────────────────────────────────────────────────────────────────
def run_single_kfold(X_data, y_data, model_type, params,
                     X_data_raw=None, Qg_data=None, Qw_data=None,
                     X_colloc=None, ya4_colloc=None,
                     n_folds=SINGLE_KFOLD_FOLDS, random_state=GLOBAL_SEED,
                     device=DEVICE, return_loss_history=False,
                     return_predictions=False):
    rows=[]; lhist=[]; preds=[]
    use_pd=model_type in PARAM_DISCOVERY_MODELS
    use_cl=model_type in COLLOCATION_MODELS
    for fid,(tr,te) in enumerate(create_fold_indices(X_data,n_folds,random_state),1):
        kw={}
        if use_pd or use_cl:
            kw.update({"X_train_raw": X_data_raw[tr] if X_data_raw is not None else None,
                       "X_test_raw":  X_data_raw[te] if X_data_raw is not None else None,
                       "Qg_train_raw":Qg_data[tr]    if Qg_data    is not None else None,
                       "Qw_train_raw":Qw_data[tr]    if Qw_data    is not None else None,
                       "Qg_test_raw": Qg_data[te]    if Qg_data    is not None else None,
                       "Qw_test_raw": Qw_data[te]    if Qw_data    is not None else None})
        if use_cl:
            kw.update({"X_colloc":X_colloc,"ya4_colloc":ya4_colloc})
        res=train_one_split(X_data[tr],y_data[tr],X_data[te],y_data[te],
                            model_type=model_type,params=params,
                            seed=random_state+fid,device=device,
                            return_loss_history=return_loss_history,
                            return_predictions=return_predictions,**kw)
        m=res["metrics"]; m["fold"]=fid; rows.append(m)
        if return_loss_history: lhist.append({"fold":fid,"loss_naoh":res["loss_naoh"],"loss_co2":res["loss_co2"]})
        if return_predictions:  preds.append({"fold":fid,"te_idx":te,"te_pred":res["te_pred"],"te_true":res["te_true"]})
    out={"df":pd.DataFrame(rows)}
    if return_loss_history: out["loss_histories"]=lhist
    if return_predictions:  out["predictions"]=preds
    return out


# ─────────────────────────────────────────────────────────────────────────────
# 100 Repeated k-fold (replaces bootstrap — used for evaluation + stats)
# ─────────────────────────────────────────────────────────────────────────────
def run_repeated_kfold(X_data, y_data, model_type, params,
                       X_data_raw=None, Qg_data=None, Qw_data=None,
                       X_colloc=None, ya4_colloc=None,
                       n_repeats=REPEATED_KFOLD_REPEATS,
                       n_folds=SINGLE_KFOLD_FOLDS,
                       base_seed=GLOBAL_SEED, device=DEVICE):
    """
    Runs n_repeats independent k-fold evaluations on different random seeds.
    Each repeat produces n_folds rows → total n_repeats × n_folds rows.
    Used for model selection, statistical testing, and uncertainty quantification.
    """
    all_rows = []
    for rep in range(n_repeats):
        split_seed = base_seed + rep
        res = run_single_kfold(
            X_data, y_data, model_type=model_type, params=params,
            X_data_raw=X_data_raw, Qg_data=Qg_data, Qw_data=Qw_data,
            X_colloc=X_colloc, ya4_colloc=ya4_colloc,
            n_folds=n_folds, random_state=split_seed, device=device)
        df_rep = res["df"].copy()
        df_rep["repeat"]     = rep + 1
        df_rep["split_seed"] = split_seed
        all_rows.append(df_rep)
        if (rep + 1) % 10 == 0 or (rep + 1) == n_repeats:
            print(f"    {rep+1}/{n_repeats}", end="\r", flush=True)
    return pd.concat(all_rows, ignore_index=True)



def average_repeated_folds(rdf):
    """Average per-fold metrics within each repeat to get one row per repeat."""
    skip = {"fold","repeat","split_seed","epochs"}
    mcols = [c for c in rdf.columns if c not in skip]
    agg = {c:"mean" for c in mcols}
    if "split_seed" in rdf.columns: agg["split_seed"] = "first"
    return rdf.groupby("repeat", as_index=False).agg(agg)


# ─────────────────────────────────────────────────────────────────────────────
# Kinetic parameter summary across repeated k-fold
# ─────────────────────────────────────────────────────────────────────────────
def kinetic_param_summary(rep_df):
    """
    Compute mean ± std for discovered kinetic parameters kga, kla, Kga
    across all repeated k-fold runs. Only applicable to _PD variants.
    """
    result = {}
    for col, label in [("final_kga","kga"),("final_kla","kla"),("final_Kga","Kga")]:
        if col not in rep_df.columns: continue
        vals = rep_df[col].dropna().values.astype(float)
        if len(vals) == 0: continue
        result[label] = {
            "mean": round(float(vals.mean()), 8),
            "std":  round(float(vals.std(ddof=1)), 8),
            "n":    len(vals)
        }
    return result


# ─────────────────────────────────────────────────────────────────────────────
# Statistical significance — paired t-test + Cohen's d
# ─────────────────────────────────────────────────────────────────────────────
def paired_ttest_cohens_d(df_a, df_b, a_name, b_name, metric_col):
    """
    Paired t-test between two models on matched repeated k-fold scores.
    Uses per-repeat means so each repeat contributes one paired observation.

    Reports:
      - Paired t-test p-value (assumes normality — valid by CLT at 100 repeats)
      - Shapiro-Wilk normality check on the differences
      - Cohen's d effect size on the differences
      - % improvement of the better model
    """
    a_vals = df_a[metric_col].dropna().values.astype(float)
    b_vals = df_b[metric_col].dropna().values.astype(float)
    n = min(len(a_vals), len(b_vals))
    a_vals = a_vals[:n]; b_vals = b_vals[:n]

    diff = a_vals - b_vals
    sh_p = float(shapiro(diff).pvalue) if len(diff) >= 3 else np.nan
    t_stat, p_val = ttest_rel(a_vals, b_vals, nan_policy='omit')
    t_stat = float(t_stat); p_val = float(p_val)

    d = float(diff.mean() / (diff.std(ddof=1) + 1e-12)) if len(diff) >= 2 else np.nan

    mean_a = float(np.mean(a_vals)); std_a = float(np.std(a_vals, ddof=1))
    mean_b = float(np.mean(b_vals)); std_b = float(np.std(b_vals, ddof=1))

    better = a_name if mean_a < mean_b else b_name
    pct_improve = ((max(mean_a, mean_b) - min(mean_a, mean_b)) / (max(mean_a, mean_b) + 1e-12)) * 100

    return {
        "metric": metric_col,
        "A": a_name,
        "B": b_name,
        "mean_A": mean_a,
        "std_A": std_a,
        "mean_B": mean_b,
        "std_B": std_b,
        "t_stat": t_stat,
        "p_value": p_val,
        "shapiro_p": sh_p,
        "significant": bool(p_val < 0.05),
        "cohens_d": d,
        "pct_improvement": float(pct_improve),
        "better_model": better,
    }

print("✅ Optuna + repeated k-fold + paired t-test + Cohen's d loaded")


✅ Optuna + repeated k-fold + paired t-test + Cohen's d loaded


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# Publication-ready plotting helpers
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.ticker as mticker

plt.rcParams.update({
    "font.family":       "serif",
    "font.size":         11,
    "axes.titlesize":    12,
    "axes.labelsize":    11,
    "xtick.labelsize":   10,
    "ytick.labelsize":   10,
    "legend.fontsize":   9,
    "figure.dpi":        150,
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

HATCH_MAP = {
    "NODE_ED":        "",
    "NODE_ED_UNC":    "////",
    "RNN":            "\\\\\\\\",
    "NODE_ED_PD":     "xxxx",
    "NODE_ED_UNC_PD": "....",
    "RNN_PD":         "oooo",
    "NODE_ED_CL":     "----",
    "NODE_ED_UNC_CL": "++++",
    "RNN_CL":         "||||",
}

SHORT_NAMES = {
    "NODE_ED":        "NODE-ED",
    "NODE_ED_UNC":    "NODE-ED\nUnc.",
    "RNN":            "RNN",
    "NODE_ED_PD":     "NODE-ED\nPD",
    "NODE_ED_UNC_PD": "NODE-ED\nUnc.PD",
    "RNN_PD":         "RNN\nPD",
    "NODE_ED_CL":     "NODE-ED\nCL",
    "NODE_ED_UNC_CL": "NODE-ED\nUnc.CL",
    "RNN_CL":         "RNN\nCL",
}


# ─────────────────────────────────────────────────────────────────────────────
# 1. Publication MSE bar chart — all models, error bars = standard deviation
# ─────────────────────────────────────────────────────────────────────────────
def plot_mse_bar(model_outputs, metric="test_agg_mse", fname_suffix="",
                 title="Test MSE — All Models", out_dir=RESULTS_DIR):
    """
    Single publication-ready bar chart comparing all models.
    Error bars = ±1 standard deviation across repeated k-fold repeats.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    models = list(model_outputs.keys())
    means  = []; stds = []

    for mt in models:
        rep_df = model_outputs[mt]["repeated"]
        col    = f"{metric}_mean" if f"{metric}_mean" in rep_df.columns else metric
        if col not in rep_df.columns:
            means.append(np.nan); stds.append(0); continue
        v = rep_df[col].dropna().values.astype(float)
        means.append(float(v.mean()))
        stds.append(float(v.std(ddof=1)))

    x  = np.arange(len(models)); bw = 0.6
    fig, ax = plt.subplots(figsize=(max(9, 1.4*len(models)), 5.5),
                           constrained_layout=True)

    ymax = max(m+s for m,s in zip(means,stds) if np.isfinite(m))

    for i, (mt, mu, sd) in enumerate(zip(models, means, stds)):
        ax.bar(x[i], mu, width=bw, facecolor="white", edgecolor="black",
               linewidth=1.1, hatch=HATCH_MAP.get(mt,""), zorder=3,
               yerr=[[sd],[sd]], capsize=5,
               error_kw={"linewidth":1.3,"ecolor":"black","capthick":1.3})
        ax.text(x[i], mu + sd + ymax*0.012,
                f"{mu:.4f}", ha="center", va="bottom", fontsize=8.5)

    ax.set_xticks(x)
    ax.set_xticklabels([SHORT_NAMES.get(m,m) for m in models],
                       rotation=0, ha="center", fontsize=10)
    ax.set_ylabel(metric.replace("test_","").replace("_"," ").upper(), fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.grid(axis="y", alpha=0.3, linewidth=0.7, zorder=0)
    ax.set_ylim(0, ymax * 1.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    safe = metric.replace("/","_")
    p = out_dir / f"comparison_{safe}{fname_suffix}.png"
    fig.savefig(p, dpi=200, bbox_inches="tight"); plt.close(fig)
    print(f"  MSE bar chart → {p.name}")


# ─────────────────────────────────────────────────────────────────────────────
# 2. Training loss curves — single k-fold, one per fold
# ─────────────────────────────────────────────────────────────────────────────
def plot_training_loss(loss_histories, model_type, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    n = len(loss_histories)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4), constrained_layout=True)
    if n == 1: axes = [axes]
    fig.suptitle(f"{model_type} — Training Loss per Fold",
                 fontsize=12, fontweight="bold")
    for ax, lh in zip(axes, loss_histories):
        naoh = lh["loss_naoh"]; co2 = lh["loss_co2"]
        comb = [a+b for a,b in zip(naoh,co2)]
        ep   = np.arange(1, len(naoh)+1)
        ax.plot(ep, naoh, label="NaOH",     color="#2196F3", linewidth=1.4)
        ax.plot(ep, co2,  label="CO₂",      color="#F44336", linewidth=1.4)
        ax.plot(ep, comb, label="Combined", color="black",   linewidth=1.4, linestyle="--")
        ax.set_title(f"Fold {lh['fold']}", fontsize=10)
        ax.set_xlabel("Epoch"); ax.set_ylabel("MSE (normalised)")
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    p = out_dir / f"{model_type}_training_loss.png"
    fig.savefig(p, dpi=200, bbox_inches="tight"); plt.close(fig)
    print(f"  Training loss → {p.name}")


# ─────────────────────────────────────────────────────────────────────────────
# 3. Concentration profiles — single k-fold, standardised axes
# ─────────────────────────────────────────────────────────────────────────────
def plot_concentration_profiles(predictions, model_type, out_dir,
                                global_naoh_ylim=None, global_co2_ylim=None):
    out_dir.mkdir(parents=True, exist_ok=True)
    sp_info = [
        (0, "NaOH outlet concentration", "#2196F3", global_naoh_ylim),
        (1, "CO₂ outlet concentration",  "#F44336", global_co2_ylim),
    ]
    gs = 0
    for pd_dict in predictions:
        fold = pd_dict["fold"]
        tep  = pd_dict["te_pred"]; tet = pd_dict["te_true"]
        for s in range(tep.shape[0]):
            gs += 1
            for si, sl, clr, ylim in sp_info:
                fig, ax = plt.subplots(figsize=(5.5, 4.2), constrained_layout=True)
                ax.plot(TIME_GRID, tet[s,:,si], color="black",  lw=2, ls="--",
                        marker="s", ms=6, label="Experimental")
                ax.plot(TIME_GRID, tep[s,:,si], color=clr, lw=2, ls="-",
                        marker="o", ms=6, label=f"{model_type} prediction")
                ax.set_xlabel("Time (min)", fontsize=11)
                ax.set_ylabel(sl, fontsize=11)
                ax.set_title(f"{model_type} | Fold {fold} | Sample {gs}", fontsize=10)
                ax.set_xticks(TIME_GRID)
                if ylim: ax.set_ylim(*ylim)
                ax.legend(fontsize=9); ax.grid(alpha=0.3)
                ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
                sp_tag = "NaOH" if si==0 else "CO2"
                fn = out_dir / f"{model_type}_fold{fold}_{sp_tag}_sample{gs:02d}.png"
                fig.savefig(fn, dpi=200, bbox_inches="tight"); plt.close(fig)
    print(f"  Concentration profiles (std axis) → {out_dir}")


# ─────────────────────────────────────────────────────────────────────────────
# 4. Kinetic parameter change — initial vs discovered, error bars = std
# ─────────────────────────────────────────────────────────────────────────────
def plot_kinetic_param_changes(kinetic_rows, out_dir):
    if not kinetic_rows: return
    out_dir.mkdir(parents=True, exist_ok=True)
    initials = {"kga": KGA_INIT, "kla": KLA_INIT, "Kga": KGA_OVR_INIT}
    param_labels = {"kga": "kga", "kla": "kla", "Kga": "Kga (overall)"}
    by_model = {}
    for row in kinetic_rows:
        mt = row["model_type"]; param = row["parameter"]
        by_model.setdefault(mt, {})[param] = row
    params = ["kga","kla","Kga"]
    for mt in by_model:
        fig, axes = plt.subplots(1, len(params), figsize=(4*len(params), 5),
                                 constrained_layout=True)
        fig.suptitle(f"{mt} — Kinetic Parameter Discovery\n(Initial vs Discovered ±1 SD)",
                     fontsize=11, fontweight="bold")
        for ax, param in zip(axes, params):
            init_val  = initials[param]
            disc_data = by_model[mt].get(param, {})
            disc_mean = disc_data.get("mean", np.nan)
            disc_std  = disc_data.get("std",  0.0)
            ax.bar(0, init_val, width=0.35, facecolor="white",
                   edgecolor="black", linewidth=1.1, hatch="////", label="Initial")
            ax.bar(0.45, disc_mean, width=0.35, facecolor="white",
                   edgecolor="black", linewidth=1.1, hatch="xxxx",
                   yerr=[[disc_std],[disc_std]], capsize=5,
                   error_kw={"linewidth":1.2,"ecolor":"black","capthick":1.2},
                   label="Discovered")
            pct = ((disc_mean - init_val) / (abs(init_val)+1e-20)) * 100
            ax.set_xticks([0, 0.45])
            ax.set_xticklabels(["Initial","Discovered"], fontsize=10)
            ax.set_ylabel(f"{param_labels[param]}", fontsize=10)
            ax.set_title(f"{param_labels[param]}\n{pct:+.1f}%", fontsize=10)
            ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
            ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
        p = out_dir / f"{mt}_kinetic_param_changes.png"
        fig.savefig(p, dpi=200, bbox_inches="tight"); plt.close(fig)
        print(f"  Kinetic param changes → {p.name}")


# ─────────────────────────────────────────────────────────────────────────────
# 5. Data efficiency MSE bar charts — one per model, error bars = std
# ─────────────────────────────────────────────────────────────────────────────
def plot_data_efficiency(ablation_df, out_dir, metric="test_agg_mse"):
    out_dir.mkdir(parents=True, exist_ok=True)
    models    = ablation_df["model_type"].unique()
    mean_col  = f"{metric}_mean"
    std_col   = f"{metric}_std"

    for mt in models:
        sub = ablation_df[ablation_df["model_type"]==mt].sort_values(
            "n_samples", ascending=False)
        if mean_col not in sub.columns: continue
        ns    = sub["n_samples"].values
        means = sub[mean_col].values
        stds  = sub[std_col].values if std_col in sub.columns else np.zeros_like(means)

        fig, ax = plt.subplots(figsize=(6, 4.5), constrained_layout=True)
        x = np.arange(len(ns))
        for i, (n, mu, sd) in enumerate(zip(ns, means, stds)):
            ax.bar(x[i], mu, width=0.55, facecolor="white", edgecolor="black",
                   linewidth=1.1, hatch=HATCH_MAP.get(mt,""), zorder=3,
                   yerr=[[sd],[sd]], capsize=5,
                   error_kw={"linewidth":1.2,"ecolor":"black","capthick":1.2})
            ax.text(x[i], mu+sd+means.max()*0.015, f"{mu:.4f}",
                    ha="center", va="bottom", fontsize=8.5)
        ax.set_xticks(x)
        ax.set_xticklabels([f"N={n}" for n in ns], fontsize=10)
        ax.set_ylabel(metric.replace("test_","").replace("_"," ").upper(), fontsize=11)
        ax.set_title(f"{SHORT_NAMES.get(mt,mt)} — Data Efficiency\n±1 SD error bars",
                     fontsize=11, fontweight="bold")
        ax.set_ylim(0, max(means+stds)*1.25)
        ax.grid(axis="y", alpha=0.3, zorder=0)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
        p = out_dir / f"{mt}_data_efficiency_{metric}.png"
        fig.savefig(p, dpi=200, bbox_inches="tight"); plt.close(fig)
        print(f"  Data efficiency → {p.name}")

print("✅ Plotting helpers loaded")


✅ Plotting helpers loaded


In [9]:
# Utilities used by the main pipeline

# Metric summary (mean ± std, no CI)
def metric_summary(df, tag):
    row = {"eval_tag": tag}
    skip = {"fold", "repeat", "split_seed", "epochs"}
    for col in df.columns:
        if col in skip:
            continue
        try:
            v = df[col].dropna().values.astype(float)
            row[f"{col}_mean"] = float(v.mean())
            row[f"{col}_std"]  = float(v.std(ddof=1)) if len(v) > 1 else 0.0
        except Exception:
            pass
    return row


def count_parameters(model):
    tot = sum(p.numel() for p in model.parameters())
    trn = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total_params": tot, "trainable_params": trn}


def profile_compute_cost(model, model_type, n_warmup=10, n_runs=50, device=None):
    """Approx compute-cost proxy: param memory + inference latency + optional FLOPs."""
    import time as _time
    model.eval()
    arch = _base_arch(model_type)
    if device is None:
        device = globals().get("DEVICE", "cpu")
    dev = torch.device(device) if not isinstance(device, torch.device) else device
    dx = torch.randn(1, N_INPUT, device=dev)
    te = T_EVAL.to(dev)
    is_node = arch in ("NODE_ED", "NODE_ED_UNC")

    with torch.no_grad():
        for _ in range(int(n_warmup)):
            _ = model(dx, te) if is_node else model(dx)

    # Prefer torch's benchmark utility if available (more stable)
    inf_ms = None
    try:
        import torch.utils.benchmark as benchmark
        tmr = (benchmark.Timer("model(dx, te)", globals={"model": model, "dx": dx, "te": te})
               if is_node else
               benchmark.Timer("model(dx)", globals={"model": model, "dx": dx}))
        r = tmr.timeit(int(n_runs))
        inf_ms = float(r.mean) * 1e3
    except Exception:
        # Fallback timing
        t0 = _time.perf_counter()
        with torch.no_grad():
            for _ in range(int(n_runs)):
                _ = model(dx, te) if is_node else model(dx)
        inf_ms = (_time.perf_counter() - t0) / max(int(n_runs), 1) * 1e3

    pi = count_parameters(model)
    flops = -1
    try:
        from torch.profiler import profile, ProfilerActivity, record_function
        with profile(activities=[ProfilerActivity.CPU], with_flops=True) as pr:
            with record_function("inference"):
                with torch.no_grad():
                    _ = model(dx, te) if is_node else model(dx)
        flops = sum(e.flops for e in pr.key_averages() if getattr(e, "flops", 0))
    except Exception:
        flops = -1

    return {
        "total_params": pi["total_params"],
        "trainable_params": pi["trainable_params"],
        "param_memory_kb": round(pi["total_params"] * 4 / 1024, 3),
        "inference_mean_ms": round(float(inf_ms), 4),
        "flops_total": flops,
    }


print("✅ Utilities loaded: metric_summary, count_parameters, profile_compute_cost")


✅ Utilities loaded: metric_summary, count_parameters, profile_compute_cost


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# MINI TRIAL RUN (fast sanity-check)
# Runs a tiny slice of the full workflow end-to-end:
#   - load data + (optional) collocation grid
#   - Optuna (few trials, 2-fold, very small epochs)
#   - tiny repeated k-fold eval
#   - MINI data-efficiency ablation (reduced N) using the updated ablation schema
#   - saves MINI_*.csv outputs under RESULTS_DIR
# ─────────────────────────────────────────────────────────────────────────────
mini_t0 = time.perf_counter()
mini_stamp = datetime.now().strftime("%d-%H%M%S")

X_data_m, y_data_m, dataset_path_m, *_ = load_outputchange_data(verbose=False)
X_raw_m, Qg_m, Qw_m = load_physics_inputs(X_data_m, y_data_m, dataset_path_m)

try:
    Xc_m, ya4_m = load_colloc_data(verbose=False)
except FileNotFoundError:
    Xc_m = ya4_m = None

n_mini = int(min(12, len(X_data_m)))
idx = np.arange(len(X_data_m))[:n_mini]
X_data_m = X_data_m[idx]
y_data_m = y_data_m[idx]
X_raw_m  = X_raw_m[idx] if X_raw_m is not None else None
Qg_m     = Qg_m[idx]    if Qg_m is not None else None
Qw_m     = Qw_m[idx]    if Qw_m is not None else None

# ── Mini base models to test (subset of BASE_MODELS) ─────────────────────────
mini_base_models = ["NODE_ED_UNC"]   # adjust to taste — must be subset of BASE_MODELS

# ── Mini physics variants to test (one per base model) ────────────────────────
mini_phys_models = []
if Xc_m is not None:
    mini_phys_models = ["NODE_ED_UNC_PD"]
else:
    mini_phys_models = ["NODE_ED_UNC_PD"]

# Physics weight sweep for mini (same as main pipeline)

hp_cols = ["physics_weight", "colloc_weight", "kinetic_weight_decay"]

mini_rows        = []
mini_eval_folds  = {}   # tag_label → rep_folds df
mini_eval_rep    = {}   # tag_label → rep df
mini_best_params = {}   # arch → best params from mini Optuna

print("\n" + "="*80)
print(f"MINI TRIAL — Full Optuna all models | N={len(X_data_m)}")
print("="*80)

mini_models_all  = mini_base_models + [m for m in mini_phys_models
                                        if not (m in COLLOCATION_MODELS and Xc_m is None)]
mini_best_params = {}

for i, mt in enumerate(mini_models_all):
    print(f"\n[Mini Optuna] {mt}...")
    bp_m, bv_m, opt_df_m = optuna_search_model(
        X_data_m, y_data_m, model_type=mt,
        X_data_raw=X_raw_m, Qg_data=Qg_m, Qw_data=Qw_m,
        X_colloc=Xc_m, ya4_colloc=ya4_m,
        n_trials=2, n_folds=2, seed=GLOBAL_SEED + 90000 + i,
        device=DEVICE, max_epochs_choices=[10])
    mini_best_params[mt] = dict(bp_m)
    tag_m = f"MINI_{mt}_{mini_stamp}"
    opt_df_m.to_csv(RESULTS_DIR/f"{tag_m}_optuna_trials.csv", index=False)
    pd.DataFrame([{**bp_m}]).to_csv(RESULTS_DIR/f"{tag_m}_optuna_best_params.csv", index=False)

print("\n[Mini Eval] 2 repeats × 2 folds for all models...")
for mt in mini_models_all:
    bp        = dict(mini_best_params[mt])
    tag_label = mt
    use_pd    = mt in PARAM_DISCOVERY_MODELS
    use_cl    = mt in COLLOCATION_MODELS
    kw_m = {}
    if use_pd or use_cl:
        kw_m.update({"X_train_raw": X_raw_m, "X_test_raw": X_raw_m,
                     "Qg_train_raw": Qg_m, "Qw_train_raw": Qw_m,
                     "Qg_test_raw": Qg_m, "Qw_test_raw": Qw_m})
    if use_cl:
        kw_m.update({"X_colloc": Xc_m, "ya4_colloc": ya4_m})
    rep_folds = run_repeated_kfold(
        X_data_m, y_data_m, model_type=mt, params=bp,
        X_data_raw=X_raw_m, Qg_data=Qg_m, Qw_data=Qw_m,
        X_colloc=Xc_m, ya4_colloc=ya4_m,
        n_repeats=2, n_folds=2, base_seed=GLOBAL_SEED + 123, device=DEVICE)
    rep = average_repeated_folds(rep_folds)
    mini_eval_folds[tag_label] = rep_folds
    mini_eval_rep[tag_label]   = rep
    mse_col  = "test_agg_mse_mean" if "test_agg_mse_mean" in rep.columns else "test_agg_mse"
    mean_mse = float(rep[mse_col].mean()) if mse_col in rep.columns else np.nan
    use_pd_e = mt in PARAM_DISCOVERY_MODELS
    pw_key   = "physics_weight" if use_pd_e else "colloc_weight"
    tag = f"MINI_{tag_label}_{mini_stamp}"
    rep_folds.to_csv(RESULTS_DIR/f"{tag}_repeated_kfold_folds.csv", index=False)
    rep.to_csv(RESULTS_DIR/f"{tag}_repeated_kfold.csv", index=False)
    mini_rows.append({"model_type": tag_label, "N": len(X_data_m),
                      "physics_weight": float(bp.get(pw_key, np.nan)),
                      "mean_test_agg_mse": mean_mse,
                      **{k: bp.get(k, np.nan) for k in hp_cols}})

mini_df = pd.DataFrame(mini_rows)
mini_df.to_csv(RESULTS_DIR/f"MINI_{mini_stamp}_summary.csv", index=False)

# ─────────────────────────────────────────────────────────────────────────────
# MINI DATA EFFICIENCY ABLATION (fast)
# Uses the same schema as the full ablation cell (incl. hp_cols), but with
# tiny repeats/folds so it can run quickly.
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("MINI DATA EFFICIENCY ABLATION")
print("="*80)

rng_abl_m = np.random.default_rng(GLOBAL_SEED + 8888)

# Choose a few valid sizes <= N, in descending order (always include full N)
N_full = int(len(X_data_m))
mini_sizes = [N_full, max(8, N_full - 2), max(6, N_full - 4)]
mini_sizes = sorted({n for n in mini_sizes if 4 <= n <= N_full}, reverse=True)

# Draw independent subsets ONCE per N — all from the original N_full samples
# Each size draws independently from the full mini dataset (same design as full ablation).
# All models at same N use identical subset — drawn before the model loop.
_mini_reduced = sorted([n for n in mini_sizes if n < N_full], reverse=True)
mini_fixed_subsets = {
    n_s: np.sort(rng_abl_m.choice(N_full, size=n_s, replace=False))
    for n_s in _mini_reduced
}

ablation_rows_m = []

for mt in list(mini_best_params.keys()):
    bp = mini_best_params[mt]
    use_pd = mt in PARAM_DISCOVERY_MODELS
    use_cl = mt in COLLOCATION_MODELS

    if use_cl and Xc_m is None:
        print(f"\n  ⚠ Skipping mini ablation for {mt} — collocation grid not loaded")
        continue

    print(f"\n  {mt} | sizes={mini_sizes}")

    # Reduced sizes (skip full N here; add it from cached evaluation below)
    for n_s in [n for n in mini_sizes if n < N_full]:
        print(f"    N={n_s}...", end=" ", flush=True)
        sub = mini_fixed_subsets[n_s]   # same subset for all models

        Xs = X_data_m[sub]; ys = y_data_m[sub]
        Xr = X_raw_m[sub] if X_raw_m is not None else None
        Qg_s = Qg_m[sub] if Qg_m is not None else None
        Qw_s = Qw_m[sub] if Qw_m is not None else None

        abl_rep_folds = run_repeated_kfold(
            Xs, ys, model_type=mt, params=bp,
            X_data_raw=Xr, Qg_data=Qg_s, Qw_data=Qw_s,
            X_colloc=Xc_m, ya4_colloc=ya4_m,
            n_repeats=2, n_folds=2, base_seed=GLOBAL_SEED + 456, device=DEVICE)
        abl_rep = average_repeated_folds(abl_rep_folds)

        row = {
            "model_type": mt,
            "n_samples": int(n_s),
            "use_pd": bool(use_pd),
            "use_cl": bool(use_cl),
            "n_repeats_completed": int(len(abl_rep)),
            **{c: bp.get(c, np.nan) for c in hp_cols},
        }
        for met in ["test_agg_mse", "test_naoh_mse", "test_co2_mse", "test_agg_mape"]:
            col = f"{met}_mean" if f"{met}_mean" in abl_rep.columns else met
            if col not in abl_rep.columns:
                continue
            v = abl_rep[col].dropna().values.astype(float)
            row[f"{met}_mean"] = float(v.mean())
            row[f"{met}_std"]  = float(v.std(ddof=1)) if len(v) > 1 else 0.0

        ablation_rows_m.append(row)

        abl_tag = f"MINI_{mt}_{mini_stamp}_ablation_N{n_s}"
        abl_rep_folds.to_csv(RESULTS_DIR/f"{abl_tag}_repeated_kfold_folds.csv", index=False)
        abl_rep.to_csv(RESULTS_DIR/f"{abl_tag}_repeated_kfold.csv", index=False)
        print("done")

    # Full-size row from cached mini evaluation
    rep_full = mini_eval_rep[mt]
    row_full = {
        "model_type": mt,
        "n_samples": int(N_full),
        "use_pd": bool(use_pd),
        "use_cl": bool(use_cl),
        "n_repeats_completed": int(len(rep_full)),
        **{c: bp.get(c, np.nan) for c in hp_cols},
    }
    for met in ["test_agg_mse", "test_naoh_mse", "test_co2_mse", "test_agg_mape"]:
        col = f"{met}_mean" if f"{met}_mean" in rep_full.columns else met
        if col not in rep_full.columns:
            continue
        v = rep_full[col].dropna().values.astype(float)
        row_full[f"{met}_mean"] = float(v.mean())
        row_full[f"{met}_std"]  = float(v.std(ddof=1)) if len(v) > 1 else 0.0

    ablation_rows_m.append(row_full)

    # Save full-size CSVs with the ablation naming convention too
    abl_tag_full = f"MINI_{mt}_{mini_stamp}_ablation_N{N_full}"
    mini_eval_folds[mt].to_csv(RESULTS_DIR/f"{abl_tag_full}_repeated_kfold_folds.csv", index=False)
    rep_full.to_csv(RESULTS_DIR/f"{abl_tag_full}_repeated_kfold.csv", index=False)

mini_ablation_df = pd.DataFrame(ablation_rows_m)
if not mini_ablation_df.empty:
    mini_ablation_df = mini_ablation_df.sort_values(["model_type", "n_samples"], ascending=[True, False]).reset_index(drop=True)
mini_ablation_df.to_csv(RESULTS_DIR/f"MINI_{mini_stamp}_data_efficiency.csv", index=False)

print(f"\nMINI TRIAL DONE — {time.perf_counter()-mini_t0:.1f}s")
print(f"Saved: MINI_{mini_stamp}_summary.csv")
print(f"Saved: MINI_{mini_stamp}_data_efficiency.csv")

display(mini_df)
display(mini_ablation_df)



MINI TRIAL — Full Optuna all models | N=12

[Mini Optuna] NODE_ED_UNC...

[Mini Optuna] NODE_ED_UNC_CL...

[Mini Eval] 2 repeats × 2 folds for all models...
    2/2
MINI DATA EFFICIENCY ABLATION

  NODE_ED_UNC | sizes=[12, 10, 8]
doneN=10...     2/2
doneN=8...     2/2

  NODE_ED_UNC_CL | sizes=[12, 10, 8]
doneN=10...     2/2
doneN=8...     2/2

MINI TRIAL DONE — 3.8s
Saved: MINI_21-235345_summary.csv
Saved: MINI_21-235345_data_efficiency.csv


,model_type,N,physics_weight,mean_test_agg_mse,colloc_weight,kinetic_weight_decay
0,NODE_ED_UNC,12,NaN,1.305071,NaN,NaN
1,NODE_ED_UNC_CL,12,NaN,0.918767,0.057159,NaN


,model_type,n_samples,use_pd,use_cl,n_repeats_completed,physics_weight,colloc_weight,kinetic_weight_decay,test_agg_mse_mean,test_agg_mse_std,test_naoh_mse_mean,test_naoh_mse_std,test_co2_mse_mean,test_co2_mse_std,test_agg_mape_mean,test_agg_mape_std
0,NODE_ED_UNC,12,False,False,2,NaN,NaN,NaN,1.305071,0.173349,0.726040,0.261377,1.884102,0.608074,409.610920,366.350436
1,NODE_ED_UNC,10,False,False,2,NaN,NaN,NaN,0.346690,0.093369,0.140678,0.011189,0.552703,0.175550,98.478908,44.701712
2,NODE_ED_UNC,8,False,False,2,NaN,NaN,NaN,8.911951,11.782041,1.299770,1.520815,16.524133,22.043268,108.123557,91.353130
3,NODE_ED_UNC_CL,12,False,True,2,NaN,0.057159,NaN,0.918767,0.221410,0.498371,0.288535,1.339162,0.731355,288.599855,259.560386
4,NODE_ED_UNC_CL,10,False,True,2,NaN,0.057159,NaN,0.370348,0.102036,0.199300,0.030302,0.541396,0.173770,97.614541,52.403138
5,NODE_ED_UNC_CL,8,False,True,2,NaN,0.057159,NaN,7.655310,10.057388,1.043037,1.195926,14.267584,18.918850,80.552320,56.289230


In [ ]:
pipeline_t0 = time.perf_counter()
X_data,y_data,dataset_path,input_cols,output_cols = load_outputchange_data(verbose=True)
X_data_raw, Qg_data, Qw_data = load_physics_inputs(X_data, y_data, dataset_path)

try:
    X_colloc, ya4_colloc = load_colloc_data(verbose=True)
    print(f"  Collocation grid: {X_colloc.shape[0]} rows")
except FileNotFoundError as e:
    print(f"  ⚠ {e}")
    print("  _CL variants will be skipped.")
    X_colloc = ya4_colloc = None

run_stamp     = datetime.now().strftime("%d-%H%M")
all_summaries = []
all_stats     = []
all_kinetic   = []
model_outputs     = {}
model_boot_preds  = {}   # stores predictions for concentration profile plots

best_params_by_model = {}
best_value_by_model  = {}

# ══════════════════════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────
# Helper: run full evaluation for one model (repeated k-fold + plots + CSVs)
# ─────────────────────────────────────────────────────────────────────────────
def run_model_evaluation(model_type, best_params, tag_label):
    """Run repeated k-fold, single k-fold, cost profiling and save all CSVs."""
    mt0    = time.perf_counter()
    use_pd = model_type in PARAM_DISCOVERY_MODELS
    use_cl = model_type in COLLOCATION_MODELS
    arch   = _base_arch(model_type)

    print("\n" + "#"*80)
    print(f"  MODEL: {tag_label}  (arch={arch}, PD={use_pd}, CL={use_cl})")
    print(f"  params: {best_params}")
    print("#"*80)

    dummy = build_model(arch, best_params)
    pi    = count_parameters(dummy); del dummy
    extra = 3 if use_pd else 0
    print(f"  Parameters: {pi['total_params']:,} model + {extra} kinetic")

    # ── Repeated k-fold ────────────────────────────────────────────────────────
    print(f"\n[2] Repeated k-fold ({REPEATED_KFOLD_REPEATS} repeats × {SINGLE_KFOLD_FOLDS} folds)...")
    rep_fold_df = run_repeated_kfold(
        X_data, y_data, model_type=model_type, params=best_params,
        X_data_raw=X_data_raw, Qg_data=Qg_data, Qw_data=Qw_data,
        X_colloc=X_colloc, ya4_colloc=ya4_colloc,
        n_repeats=REPEATED_KFOLD_REPEATS, n_folds=SINGLE_KFOLD_FOLDS,
        base_seed=GLOBAL_SEED, device=DEVICE)
    rep_df = average_repeated_folds(rep_fold_df)
    print(f"\n    Completed: {len(rep_df)}/{REPEATED_KFOLD_REPEATS} repeats")
    for met in ["test_agg_mse","test_naoh_mse","test_co2_mse"]:
        col = f"{met}_mean" if f"{met}_mean" in rep_df.columns else met
        if col in rep_df.columns:
            v = rep_df[col].dropna().values.astype(float)
            print(f"    {met}: {v.mean():.5f} ± {v.std(ddof=1):.5f}")

    # ── Kinetic param summary (PD only) ────────────────────────────────────────
    kp_sum = {}
    if use_pd:
        kp_sum = kinetic_param_summary(rep_fold_df)
        for pname, stats in kp_sum.items():
            init_val = {"kga":KGA_INIT,"kla":KLA_INIT,"Kga":KGA_OVR_INIT}.get(pname,np.nan)
            pct_chg  = ((stats["mean"]-init_val)/abs(init_val+1e-20))*100
            print(f"    {pname}: {init_val:.4e} → {stats['mean']:.4e} ({pct_chg:+.1f}%)")
            all_kinetic.append({"model_type":tag_label,"parameter":pname,
                                 "initial_value":init_val,"pct_change":round(pct_chg,2),
                                 **stats})

    # ── Single k-fold for plots ────────────────────────────────────────────────
    print("\n[3] Single 5-fold (plots)...")
    plot_dir = RESULTS_DIR / f"{tag_label}_{run_stamp}_plots"
    single_res = run_single_kfold(
        X_data, y_data, model_type=model_type, params=best_params,
        X_data_raw=X_data_raw, Qg_data=Qg_data, Qw_data=Qw_data,
        X_colloc=X_colloc, ya4_colloc=ya4_colloc,
        n_folds=SINGLE_KFOLD_FOLDS, random_state=PLOT_KFOLD_SEED, device=DEVICE,
        return_loss_history=True, return_predictions=True)
    plot_training_loss(single_res["loss_histories"], tag_label, plot_dir)
    model_boot_preds[tag_label] = single_res.get("predictions", [])

    # ── Compute cost ────────────────────────────────────────────────────────────
    pm   = build_model(arch, best_params)
    cost = profile_compute_cost(pm, model_type, device=DEVICE); del pm

    # ── Summary row ─────────────────────────────────────────────────────────────
    smry = metric_summary(rep_df, "repeated_kfold_100")
    smry.update({"model_type":tag_label, "arch":arch,
                 "use_pd":use_pd, "use_cl":use_cl,
                 "physics_weight": float(best_params.get("physics_weight", np.nan)) if use_pd else np.nan,
                 "colloc_weight":  float(best_params.get("colloc_weight",  np.nan)) if use_cl else np.nan,
                 "total_params":pi["total_params"], "kinetic_params":extra,
                 "param_memory_kb":cost["param_memory_kb"],
                 "inference_mean_ms":cost["inference_mean_ms"],
                 "flops_total":cost["flops_total"],
                 "n_repeats_completed":len(rep_df),
                 "model_total_time_sec":float(time.perf_counter()-mt0)})
    all_summaries.append(smry)

    # ── Store in model_outputs ─────────────────────────────────────────────────
    model_outputs[tag_label] = {
        "repeated":       rep_df,
        "repeated_folds": rep_fold_df,
        "best_params":    best_params,
        "predictions":    single_res.get("predictions", []),
    }

    # ── Save CSVs ──────────────────────────────────────────────────────────────
    tag = f"{tag_label}_{run_stamp}"
    pd.DataFrame([best_params]).to_csv(
        RESULTS_DIR/f"{tag}_best_params.csv", index=False)
    rep_fold_df.to_csv(
        RESULTS_DIR/f"{tag}_repeated_kfold_folds.csv", index=False)
    rep_df.to_csv(
        RESULTS_DIR/f"{tag}_repeated_kfold.csv", index=False)
    if single_res.get("df") is not None and len(single_res.get("df", pd.DataFrame())) > 0:
        single_res["df"].to_csv(
            RESULTS_DIR/f"{tag}_single_kfold.csv", index=False)
    pd.DataFrame([cost]).to_csv(
        RESULTS_DIR/f"{tag}_compute_cost.csv", index=False)
    if use_pd and kp_sum:
        pd.DataFrame([{"model_type":tag_label,
                       **{f"{p}_{k}":v for p,stats in kp_sum.items()
                          for k,v in stats.items()}}
                      ]).to_csv(RESULTS_DIR/f"{tag}_kinetic_params.csv", index=False)


# ══════════════════════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1 — Full Optuna for EVERY model (base + physics variants)
# Each model independently optimises ALL its hyperparameters including
# physics_weight / colloc_weight where applicable.
# ══════════════════════════════════════════════════════════════════════════════
print("="*80)
print("  PHASE 1 — Full Optuna: ALL models (base + physics variants)")
print("="*80)

best_params_by_model = {}   # model_type → best params

for mi, model_type in enumerate(MODEL_TYPES):
    if model_type in COLLOCATION_MODELS and X_colloc is None:
        print(f"\n  ⚠ Skipping {model_type} — collocation grid not loaded")
        continue

    opt_seed = GLOBAL_SEED + 1000 + 17 * mi
    print(f"\n[Optuna] {model_type}...")
    bp, bv, opt_df = optuna_search_model(
        X_data, y_data, model_type=model_type,
        X_data_raw=X_data_raw, Qg_data=Qg_data, Qw_data=Qw_data,
        X_colloc=X_colloc, ya4_colloc=ya4_colloc,
        n_trials=N_TRIALS, n_folds=OPTUNA_FOLDS, seed=opt_seed, device=DEVICE)
    best_params_by_model[model_type] = bp
    print(f"  Best val: {bv:.5f}  params: {bp}")
    tag = f"{model_type}_{run_stamp}"
    opt_df.to_csv(RESULTS_DIR/f"{tag}_optuna_trials.csv", index=False)
    pd.DataFrame([{**bp, "best_neg_mse_cv": float(bv)}]).to_csv(
        RESULTS_DIR/f"{tag}_optuna_best_params.csv", index=False)

print("\n" + "="*80)
print("  PHASE 2 — Evaluation of all models")
print("="*80)

for model_type in list(best_params_by_model.keys()):
    best_params = dict(best_params_by_model[model_type])
    run_model_evaluation(model_type, best_params, tag_label=model_type)


# ── Global axis concentration profiles ───────────────────────────────────────
print("\n[Post-loop] Concentration profiles with standardised axes...")
all_naoh = []; all_co2 = []
for mt, mdata in model_outputs.items():
    for pd_dict in mdata["predictions"]:
        all_naoh.append(pd_dict["te_pred"][:,:,0].ravel())
        all_naoh.append(pd_dict["te_true"][:,:,0].ravel())
        all_co2.append( pd_dict["te_pred"][:,:,1].ravel())
        all_co2.append( pd_dict["te_true"][:,:,1].ravel())

if all_naoh:
    naoh_all = np.concatenate(all_naoh); co2_all = np.concatenate(all_co2)
    pad_n = (naoh_all.max()-naoh_all.min())*0.08
    pad_c = (co2_all.max() -co2_all.min()) *0.08
    global_naoh_ylim = (max(0, naoh_all.min()-pad_n), naoh_all.max()+pad_n)
    global_co2_ylim  = (max(0, co2_all.min() -pad_c), co2_all.max() +pad_c)
    print(f"  NaOH axis: {global_naoh_ylim}")
    print(f"  CO2  axis: {global_co2_ylim}")
    for mt, mdata in model_outputs.items():
        if not mdata.get("predictions"):  # skip variants without plots
            continue
        pdir = RESULTS_DIR / f"{mt}_{run_stamp}_plots"
        plot_concentration_profiles(
            mdata["predictions"], mt, pdir,
            global_naoh_ylim=global_naoh_ylim,
            global_co2_ylim=global_co2_ylim)

# ── Kinetic parameter change plots ───────────────────────────────────────────
if all_kinetic:
    print("\n[Post-loop] Kinetic parameter change plots...")
    plot_kinetic_param_changes(all_kinetic, RESULTS_DIR)

# ── Build weight groups (used by bar charts and significance testing) ─────────
import re as _re
from collections import defaultdict as _dd

def _wgt(mt):
    m = _re.search(r"(_w[0-9p]+)$", mt)
    return m.group(1) if m else "_base"

weight_groups = _dd(list)
for mt in model_outputs.keys():
    weight_groups[_wgt(mt)].append(mt)

# ── Publication MSE bar charts — one per weight group ────────────────────────
print("\n[Post-loop] Publication MSE bar charts...")
for wgroup, mt_list in sorted(weight_groups.items()):
    mo_subset = {mt: model_outputs[mt] for mt in mt_list}
    wlabel = wgroup.lstrip("_")
    for metric in ["test_agg_mse","test_naoh_mse","test_co2_mse"]:
        title = (f"Test {metric.replace('test_','').replace('_',' ').upper()}"
                 f" | {wlabel}")
        plot_mse_bar(mo_subset, metric=metric, title=title,
                     out_dir=RESULTS_DIR, fname_suffix=f"_{wgroup}")

# ── Pairwise significance testing — grouped by physics weight ────────────────
print("\n[Post-loop] Pairwise significance testing (within weight groups)...")

for wgroup, mt_list in sorted(weight_groups.items()):
    print(f"\n  Weight group: {wgroup}  ({len(mt_list)} models)")
    pairs = [(a,b) for i,a in enumerate(mt_list) for b in mt_list[i+1:]]
    for a, b in pairs:
        rep_a = model_outputs[a]["repeated"]
        rep_b = model_outputs[b]["repeated"]
        for met in ["test_agg_mse","test_naoh_mse","test_co2_mse"]:
            col = f"{met}_mean" if f"{met}_mean" in rep_a.columns else met
            if col not in rep_a.columns or col not in rep_b.columns: continue
            result = paired_ttest_cohens_d(rep_a, rep_b, a, b, metric_col=col)
            result["weight_group"] = wgroup
            all_stats.append(result)
        sig = "SIGNIFICANT" if result.get("significant") else "ns"
        print(f"    {a} vs {b}: {sig} "
              f"(p={result['p_value']:.4f}, d={result['cohens_d']:.3f})")

# ── Consolidate + save ────────────────────────────────────────────────────────
all_summary_df = pd.DataFrame(all_summaries)
all_stats_df   = pd.DataFrame(all_stats)
kinetic_df     = pd.DataFrame(all_kinetic)
pipe_t         = float(time.perf_counter()-pipeline_t0)
runtime_df     = pd.DataFrame([{"run_stamp":run_stamp,
                                 "pipeline_total_time_sec":pipe_t,
                                 "n_models":len(model_outputs),
                                 "n_trials":N_TRIALS,
                                 "repeated_kfold_repeats":REPEATED_KFOLD_REPEATS}])

all_summary_df.to_csv(RESULTS_DIR/f"ALL_MODELS_{run_stamp}_summary.csv",    index=False)
all_stats_df.to_csv(  RESULTS_DIR/f"ALL_MODELS_{run_stamp}_sig_tests.csv",  index=False)
kinetic_df.to_csv(    RESULTS_DIR/f"ALL_MODELS_{run_stamp}_kinetic.csv",     index=False)
runtime_df.to_csv(    RESULTS_DIR/f"ALL_MODELS_{run_stamp}_runtime.csv",     index=False)

print("\n"+"="*80); print(f"PIPELINE DONE — {pipe_t:.1f}s"); print("="*80)
disp_cols = ["model_type","test_agg_mse_mean","test_naoh_mse_mean","test_co2_mse_mean",
             "total_params","inference_mean_ms"]
display(all_summary_df[[c for c in disp_cols if c in all_summary_df.columns]])
if not kinetic_df.empty:
    print("\nKinetic parameter summary:")
    display(kinetic_df)
display(all_stats_df[["metric","A","B","mean_A","mean_B","std_A","std_B",
                       "t_stat","p_value","significant","cohens_d",
                       "pct_improvement","better_model"]])


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DATA EFFICIENCY ABLATION  30→25→20→15 samples, repeated k-fold per size 
# Base models use Optuna-tuned params.
# Physics variants (e.g. NODE_ED_PD_w0p1) use base model arch params + fixed physics weight.
# NOTE: Requires the main pipeline cell to have been run (it populates
#       model_outputs, X_data, y_data, etc.).
# ─────────────────────────────────────────────────────────────────────────────

_required_vars = ["model_outputs", "X_data", "y_data", "X_data_raw", "Qg_data", "Qw_data",
                  "X_colloc", "ya4_colloc", "run_stamp"]
_missing = [v for v in _required_vars if v not in globals()]
if _missing:
    raise RuntimeError(
        "Ablation prerequisites missing: " + ", ".join(_missing) +
        "\nRun the main pipeline cell (PHASE 1/2) first, then rerun this ablation cell."
    )
if not isinstance(model_outputs, dict) or len(model_outputs) == 0:
    raise RuntimeError(
        "model_outputs is empty — nothing to ablate. "
        "Run the main pipeline cell (PHASE 1/2) first."
    )

print("\n"+"="*80); print("DATA EFFICIENCY ABLATION"); print("="*80)

ablation_rows = []

# ── Draw independent subsets ONCE per N — all from the original 30 samples ───
# Each N_size (25, 20, 15) draws independently from the full 30-sample dataset.
# All models at the same N use the SAME subset — ensuring pairwise statistical
# tests compare models on identical data. Subsets are drawn before the model
# loop so the rng state advances predictably regardless of model count.
rng_abl = np.random.default_rng(GLOBAL_SEED + 99999)
ablation_sizes_run = sorted([n for n in ABLATION_SIZES if n < len(X_data)], reverse=True)

fixed_subsets = {
    n_s: np.sort(rng_abl.choice(len(X_data), size=n_s, replace=False))
    for n_s in ablation_sizes_run
}
print("Independent fixed subsets per N — all drawn from full 30 (shared across models):")
for n_s in sorted(fixed_subsets.keys(), reverse=True):
    print(f"  N={n_s}: indices {fixed_subsets[n_s][:5]}... (first 5 of {n_s})")

hp_cols = ["physics_weight", "colloc_weight", "kinetic_weight_decay"]

for model_type in list(model_outputs.keys()):
    bp = model_outputs[model_type]["best_params"]
    # model_type is the plain model name (e.g. NODE_ED_PD) — no weight suffix in v9
    # model_type is plain name (e.g. NODE_ED_PD) — no weight suffix in v9
    use_pd = model_type in PARAM_DISCOVERY_MODELS
    use_cl = model_type in COLLOCATION_MODELS

    if use_cl and X_colloc is None:
        print(f"\n  ⚠ Skipping ablation for {model_type} — collocation grid not loaded")
        continue

    print(f"\n  {model_type}")

    for n_s in ablation_sizes_run:
        print(f"    N={n_s}...", end=" ", flush=True)
        sub = fixed_subsets[n_s]   # same subset for ALL models at this N
        Xs  = X_data[sub]; ys = y_data[sub]
        Xr  = X_data_raw[sub] if X_data_raw is not None else None
        Qg_s= Qg_data[sub]    if Qg_data    is not None else None
        Qw_s= Qw_data[sub]    if Qw_data    is not None else None

        abl_rep_df = run_repeated_kfold(
            Xs, ys, model_type=model_type, params=bp,
            X_data_raw=Xr, Qg_data=Qg_s, Qw_data=Qw_s,
            X_colloc=X_colloc, ya4_colloc=ya4_colloc,
            n_repeats=REPEATED_KFOLD_REPEATS,
            n_folds=min(SINGLE_KFOLD_FOLDS, max(2, n_s//4)),
            base_seed=GLOBAL_SEED, device=DEVICE)
        abl_df = average_repeated_folds(abl_rep_df)

        row = {"model_type":model_type,"n_samples":n_s,
               "use_pd":use_pd,"use_cl":use_cl,
               "n_repeats_completed":len(abl_df),
               **{c: bp.get(c, np.nan) for c in hp_cols}}
        for met in ["test_agg_mse","test_naoh_mse","test_co2_mse","test_agg_mape"]:
            col = f"{met}_mean" if f"{met}_mean" in abl_df.columns else met
            if col not in abl_df.columns: continue
            v = abl_df[col].dropna().values.astype(float)
            row[f"{met}_mean"] = round(float(v.mean()), 6)
            row[f"{met}_std"]  = round(float(v.std(ddof=1)), 6)
        ablation_rows.append(row)

        abl_tag = f"{model_type}_{run_stamp}_ablation_N{n_s}"
        abl_rep_df.to_csv(RESULTS_DIR/f"{abl_tag}_repeated_kfold_folds.csv", index=False)
        abl_df.to_csv(    RESULTS_DIR/f"{abl_tag}_repeated_kfold.csv",        index=False)
        print("\ndone")

# ── Append full-size results from the main pipeline evaluation ────────────────
# These were already computed in the main loop — no need to re-run.
for model_type in list(model_outputs.keys()):
    rep_df_full = model_outputs[model_type]["repeated"]
    row_full = {"model_type": model_type, "n_samples": len(X_data),
                "use_pd": model_type in PARAM_DISCOVERY_MODELS,
                "use_cl": model_type in COLLOCATION_MODELS,
                "n_repeats_completed": len(rep_df_full),
                **{c: model_outputs[model_type]["best_params"].get(c, np.nan) for c in hp_cols}}

    for met in ["test_agg_mse","test_naoh_mse","test_co2_mse","test_agg_mape"]:
        col = f"{met}_mean" if f"{met}_mean" in rep_df_full.columns else met
        if col not in rep_df_full.columns: continue
        v = rep_df_full[col].dropna().values.astype(float)
        row_full[f"{met}_mean"] = float(v.mean())
        row_full[f"{met}_std"]  = float(v.std(ddof=1))

    # Save full-size CSVs using same naming convention as ablation sizes
    abl_tag_full = f"{model_type}_{run_stamp}_ablation_N{len(X_data)}"
    model_outputs[model_type]["repeated_folds"].to_csv(
        RESULTS_DIR/f"{abl_tag_full}_repeated_kfold_folds.csv", index=False)
    rep_df_full.to_csv(
        RESULTS_DIR/f"{abl_tag_full}_repeated_kfold.csv", index=False)

    ablation_rows.append(row_full)

# Sort so N=full→25→20→15 order is preserved in the consolidated CSV
ablation_df = pd.DataFrame(ablation_rows)
ablation_df = ablation_df.sort_values(["model_type","n_samples"],
                                       ascending=[True,False]).reset_index(drop=True)
ablation_df.to_csv(RESULTS_DIR/f"ALL_MODELS_{run_stamp}_data_efficiency.csv", index=False)

print("\n[Ablation plots] Data efficiency MSE bar charts...")
plot_data_efficiency(ablation_df, RESULTS_DIR, metric="test_agg_mse")
plot_data_efficiency(ablation_df, RESULTS_DIR, metric="test_naoh_mse")
plot_data_efficiency(ablation_df, RESULTS_DIR, metric="test_co2_mse")

print("\n"+"="*80); print("ABLATION DONE"); print("="*80)
display(ablation_df)

# ── Pairwise statistical tests within each N_samples group ───────────────────
# For each N, compare all model pairs using paired t-test + Cohen's d.
# "Paired" means the same 100 random splits are used for both models
# (same base_seed), so the per-repeat differences isolate model effect.
print("\n[Ablation stats] Pairwise tests within each N_samples group...")

from scipy.stats import ttest_rel

def _paired_ttest_ablation(a_vals, b_vals, a_name, b_name, metric, n_samples):
    """Paired t-test + Cohen's d on two sets of per-repeat mean scores."""
    a = np.asarray(a_vals, float); b = np.asarray(b_vals, float)
    n  = min(len(a), len(b)); a, b = a[:n], b[:n]
    diff = a - b
    d    = float(diff.mean() / (diff.std(ddof=1) + 1e-12))
    t, p = ttest_rel(a, b)
    better = a_name if diff.mean() < 0 else b_name
    return {
        "n_samples":   n_samples,
        "metric":      metric,
        "A":           a_name,
        "B":           b_name,
        "mean_A":      float(a.mean()),
        "mean_B":      float(b.mean()),
        "std_A":       float(a.std(ddof=1)),
        "std_B":       float(b.std(ddof=1)),
        "mean_diff":   float(diff.mean()),
        "t_stat":      float(t),
        "p_value":     float(p),
        "significant": bool(p < 0.05),
        "cohens_d":    round(d, 4),
        "pct_improvement": round(abs(a.mean()-b.mean())/(max(a.mean(),b.mean())+1e-12)*100, 2),
        "better_model":better,
    }

abl_stat_rows = []

# Collect per-repeat means for every model at every N_samples
# Sources:
#   N=30 (full) → model_outputs[mt]["repeated"]  (already computed in pipeline)
#   N=25,20,15  → re-read from the CSVs just saved in the ablation loop above

# Build lookup: n_samples → {model_type → per-repeat-mean series}
from collections import defaultdict
n_to_model_series = defaultdict(dict)

# Full-size (N=30) — from model_outputs in memory
for mt in model_outputs.keys():
    rep_df_full = model_outputs[mt]["repeated"]
    for met in ["test_agg_mse", "test_naoh_mse", "test_co2_mse"]:
        col = f"{met}_mean" if f"{met}_mean" in rep_df_full.columns else met
        if col in rep_df_full.columns:
            n_to_model_series[len(X_data)].setdefault(mt, {})[met] = \
                rep_df_full[col].dropna().values.astype(float)

# Reduced sizes — from saved CSVs
for model_type in model_outputs.keys():
    for n_s in ABLATION_SIZES:
        if n_s >= len(X_data):
            continue
        abl_tag = f"{model_type}_{run_stamp}_ablation_N{n_s}"
        fpath = RESULTS_DIR / f"{abl_tag}_repeated_kfold.csv"
        if not fpath.exists():
            continue
        abl_df_loaded = pd.read_csv(fpath)
        for met in ["test_agg_mse", "test_naoh_mse", "test_co2_mse"]:
            col = f"{met}_mean" if f"{met}_mean" in abl_df_loaded.columns else met
            if col in abl_df_loaded.columns:
                n_to_model_series[n_s].setdefault(model_type, {})[met] = \
                    abl_df_loaded[col].dropna().values.astype(float)

# Run all pairwise tests within each N group
for n_s in sorted(n_to_model_series.keys(), reverse=True):
    model_list = sorted(n_to_model_series[n_s].keys())
    pairs = [(a, b) for i, a in enumerate(model_list) for b in model_list[i+1:]]
    print(f"\n  N={n_s}: {len(model_list)} models, {len(pairs)} pairs")
    for a, b in pairs:
        for met in ["test_agg_mse", "test_naoh_mse", "test_co2_mse"]:
            a_vals = n_to_model_series[n_s].get(a, {}).get(met)
            b_vals = n_to_model_series[n_s].get(b, {}).get(met)
            if a_vals is None or b_vals is None:
                continue
            result = _paired_ttest_ablation(a_vals, b_vals, a, b, met, n_s)
            abl_stat_rows.append(result)
        # Print summary for agg_mse
        r = next((r for r in abl_stat_rows
                  if r["n_samples"]==n_s and r["A"]==a and r["B"]==b
                  and r["metric"]=="test_agg_mse"), None)
        if r:
            sig = "SIGNIFICANT" if r["significant"] else "ns"
            print(f"    {a} vs {b}: {sig} "
                  f"(p={r['p_value']:.4f}, d={r['cohens_d']:.3f}, "
                  f"better={r['better_model']})")

abl_stats_df = pd.DataFrame(abl_stat_rows)
abl_stats_df.to_csv(RESULTS_DIR/f"ALL_MODELS_{run_stamp}_ablation_sig_tests.csv",
                    index=False)
print(f"\n  Saved: ALL_MODELS_{run_stamp}_ablation_sig_tests.csv")
if not abl_stats_df.empty:
    display(abl_stats_df[["n_samples","metric","A","B","mean_A","mean_B",
                           "t_stat","p_value","significant","cohens_d","better_model"]])


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RESULTS VISUALISATION — Clustered column charts per model family
# Each chart shows Base / PD / CL variants across N=30,25,20,15.
# Reads from model_outputs (N=30) and ablation CSVs (N=25,20,15).
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.patches as mpatches
matplotlib.rcParams['font.family'] = 'serif'

ABLATION_NS = [30, 25, 20, 15]
N_LABELS    = [f"N={n}" for n in ABLATION_NS]
METRIC_COL  = "test_agg_mse_mean"


def _get_mse_stats(mt, n_samples):
    """Return (mean, sd) for model mt at n_samples."""
    if mt not in model_outputs:
        return np.nan, np.nan
    if n_samples == len(X_data):
        rep = model_outputs[mt]["repeated"]
    else:
        pat = RESULTS_DIR / f"{mt}_{run_stamp}_ablation_N{n_samples}_repeated_kfold.csv"
        if not pat.exists():
            return np.nan, np.nan
        rep = pd.read_csv(pat)
    col = METRIC_COL if METRIC_COL in rep.columns else "test_agg_mse"
    if col not in rep.columns:
        return np.nan, np.nan
    v = rep[col].dropna().values.astype(float)
    return float(v.mean()), float(v.std(ddof=1)) if len(v) > 1 else 0.0


def _series(mt):
    """Return (means, sds) across ABLATION_NS."""
    means, sds = [], []
    for n in ABLATION_NS:
        m, s = _get_mse_stats(mt, n)
        means.append(m); sds.append(s)
    return means, sds


def _save(fig, name):
    p = RESULTS_DIR / f"viz_{name}_{run_stamp}.png"
    fig.savefig(p, dpi=150, bbox_inches="tight", facecolor="white")
    print(f"  Saved: {p.name}")


def plot_model_family(base_mt, title, fname):
    """
    Clustered column chart for one model family.
    Three variant groups: Base, PD, CL.
    Within each group: one cluster of 4 bars (N=30,25,20,15).
    Layout: x-axis = [Base-N30 Base-N25 Base-N20 Base-N15 | PD-N30 ... | CL-N30 ...]
    Colour-coded by N value; hatch-coded by variant type.
    """
    pd_mt  = f"{base_mt}_PD"
    cl_mt  = f"{base_mt}_CL"
    variants = [
        (base_mt, "Base", "",    "solid"),
        (pd_mt,   "PD",   "//",  "solid"),
        (cl_mt,   "CL",   "..",  "solid"),
    ]

    # Colour map for N values — darker = more data
    n_colours = {30: "#2c3e50", 25: "#5d6d7e", 20: "#aab7b8", 15: "#d5d8dc"}

    n_variants = len(variants)       # 3
    n_n        = len(ABLATION_NS)    # 4
    bar_w      = 0.18
    gap        = 0.35                # gap between variant groups
    group_w    = n_n * bar_w         # width of one N-cluster

    fig, ax = plt.subplots(figsize=(13, 6), dpi=150)
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    x_ticks_pos = []   # centre of each variant group
    x_tick_labs = []

    all_means = []

    for vi, (mt, label, hatch, _) in enumerate(variants):
        group_start = vi * (group_w + gap)
        centres     = [group_start + i * bar_w for i in range(n_n)]
        x_ticks_pos.append(group_start + group_w / 2 - bar_w / 2)
        x_tick_labs.append(label)

        means, sds = _series(mt)
        for ni, (n, m, s) in enumerate(zip(ABLATION_NS, means, sds)):
            if np.isnan(m):
                continue
            xpos = centres[ni]
            bar  = ax.bar(xpos, m, bar_w,
                          color=n_colours[n], hatch=hatch,
                          edgecolor="black", linewidth=1.2,
                          yerr=s, capsize=4,
                          error_kw=dict(ecolor="black", lw=1.2, capthick=1.2),
                          label=f"N={n}" if vi == 0 else "_nolegend_")
            all_means.append(m)
            # Value label above bar
            ax.text(xpos, m + (s if np.isfinite(s) else 0) + 0.008,
                    f"{m:.4f}", ha="center", va="bottom",
                    fontsize=7.5, rotation=90)

        # Vertical separator between variant groups
        if vi < n_variants - 1:
            sep_x = group_start + group_w + gap / 2
            ax.axvline(sep_x, color="gray", lw=0.8, ls="--", alpha=0.5)

    # Axes formatting
    ax.set_xticks(x_ticks_pos)
    ax.set_xticklabels(x_tick_labs, fontsize=13, fontweight="bold")
    ax.set_ylabel("Aggregated MSE (normalised)", fontsize=12)
    ax.set_title(title, fontsize=13, fontweight="bold", pad=14)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", color="gray", alpha=0.25, linewidth=0.8)
    ax.tick_params(axis="y", labelsize=11)

    # N-value colour legend
    n_legend = [mpatches.Patch(color=n_colours[n], label=f"N={n}") for n in ABLATION_NS]
    ax.legend(handles=n_legend, title="Data size", fontsize=10,
              title_fontsize=10, frameon=False,
              loc="upper right", ncol=2)

    # Y-axis upper limit with headroom for labels
    ymax = max(all_means) * 1.45 if all_means else 1.0
    ax.set_ylim(0, ymax)

    plt.tight_layout()
    _save(fig, fname)
    plt.show()


# ── Generate one chart per base model family ──────────────────────────────────
FAMILIES = [
    ("NODE_ED",    "NODE-ED — Base vs PD vs CL across Data Sizes",       "node_ed_family"),
    ("NODE_ED_UNC","NODE-ED-UNC (MTL) — Base vs PD vs CL across Data Sizes","node_unc_family"),
    ("RNN",        "RNN — Base vs PD vs CL across Data Sizes",           "rnn_family"),
    ("GRU",        "GRU — Base vs PD vs CL across Data Sizes",           "gru_family"),
]

for base_mt, title, fname in FAMILIES:
    print(f"\n[Viz] {title}")
    plot_model_family(base_mt, title, fname)

print("\n[Viz] All plots complete.")
